# Complejidad y actividad espectral del EEG a través de las etapas del sueño
### Pipeline reproducible de Ingeniería de Datos sobre Sleep-EDF Expanded (PhysioNet)

**Trabajo Práctico Final — Introducción a la Ingeniería de Datos**

Este notebook es el documento principal del trabajo. La lógica reutilizable vive en `src/` y se importa desde aquí;
el notebook ejecuta las etapas, muestra las validaciones y documenta cada decisión metodológica.

# Declaración de uso de IA

| Herramienta | Uso |
|---|---|
| OpenAI Codex | Diseño de la arquitectura del proyecto, generación y revisión de código Python y documentación técnica. |
| Claude Code — modelo Claude Opus 5.5 (`claude-opus-5-5`); versión de la herramienta: Opus 5.5 | Diseño de la arquitectura del proyecto, generación y revisión de código Python y documentación técnica. |

Todas las decisiones metodológicas fueron revisadas y quedan documentadas en este notebook. Las herramientas de IA no
generaron datos: todos los resultados provienen de la ejecución del pipeline sobre datos públicos.

## Objetivo

Construir un pipeline de datos **reproducible de punta a punta** que:

1. descarga automáticamente EEG de sueño público (EDF/EDF+);
2. lo organiza según **BIDS**;
3. lo transforma (ETL) en epochs de 30 s etiquetados con su etapa de sueño;
4. calcula métricas espectrales y de complejidad por epoch y canal;
5. detecta y documenta datos faltantes, outliers estadísticos y artefactos de señal;
6. carga los resultados en una base **SQLite transaccional normalizada (3NF)**;
7. construye un **Data Warehouse en esquema estrella** y lo consulta con SQL;
8. comunica los resultados con visualizaciones Plotly y un dashboard Streamlit.

El objetivo principal **no** es demostrar una hipótesis neurocientífica nueva, sino demostrar un pipeline de
Ingeniería de Datos usando una pregunta neurocientífica real como caso de estudio.

## Pregunta de investigación

> **¿Cómo varían la complejidad y la actividad espectral del EEG entre las distintas etapas del sueño?**

- *Actividad espectral*: potencia absoluta y relativa en las bandas delta, theta, alfa, sigma y beta.
- *Complejidad*: Sample Entropy (impredecibilidad temporal) y Spectral Entropy (dispersión de la potencia en frecuencia).

Todos los resultados son **descriptivos**: no se realizan tests de hipótesis, por lo que una diferencia observada no
implica significancia estadística, y una asociación no implica causalidad.

## Dataset

**Sleep-EDF Expanded** (Kemp et al., 2000; PhysioNet, DOI [10.13026/C2X676](https://doi.org/10.13026/C2X676)),
subconjunto *sleep-cassette* (SC): polisomnografías ambulatorias de ~20 h de personas sanas sin medicación.

| Elemento | Descripción |
|---|---|
| `SC4ssN?0-PSG.edf` | Polisomnografía: EEG Fpz-Cz y Pz-Oz (100 Hz), EOG (100 Hz), EMG, respiración, temperatura (1 Hz) |
| `SC4ssN??-Hypnogram.edf` | Etapas de sueño puntuadas por expertos (reglas de Rechtschaffen & Kales) en segmentos múltiplos de 30 s |
| `SC-subjects.xls` | Edad, sexo y hora de apagado de luces por sujeto y noche |
| `SHA256SUMS.txt` | Lista oficial de archivos con sus hashes |

`ss` = sujeto, `N` = noche. El sufijo del hipnograma (p. ej. `EC`, `EH`) **difiere** del del PSG (`E0`), por lo que
el emparejamiento se hace por (sujeto, noche) y no por nombre de archivo.

**Selección de sujetos.** Durante el desarrollo se usaron 2 sujetos. La versión final procesa **10 sujetos × 2
noches** cambiando solo la configuración (`SUBJECT_IDS` / `N_SUBJECTS` en `src/config.py`), sin modificar la lógica.

Los primeros 10 sujetos por ID resultaron ser todas mujeres de 25–34 años. Para no limitar el análisis a ese grupo,
se eligió una **muestra estratificada por sexo y edad**: para cada sexo, entre los sujetos con 2 noches, el sujeto
más cercano a los cuantiles 10/30/50/70/90 % de edad.

| Sexo | Sujetos (edad) |
|---|---|
| F | 5 (28), 20 (51), 22 (56), 40 (67), 65 (88) |
| M | 14 (27), 38 (51), 33 (60), 50 (71), 76 (90) |

## Arquitectura general

```
PhysioNet (AWS Open Data mirror / physionet.org)
    │  extract.py      descarga + SHA256 + manifest
    ▼
data/raw/            EDF originales (no se modifican)
    │  bids.py         MNE-BIDS
    ▼
data/bids/           sub-XX/ses-N/eeg/  + participants.tsv, events.tsv, channels.tsv
    │  preprocessing.py  lectura, validación, epochs, recorte, filtro
    │  features.py       Welch, potencias, entropías, RMS, pico a pico
    │  quality.py        artefactos, outliers, faltantes
    │  pipeline.py       orquestación + caché por registro
    ▼
data/processed/      features_tidy.parquet (epoch × canal), epochs, recordings, reportes
    │  database_oltp.py
    ▼
data/database/sleep_oltp.sqlite   modelo relacional 3NF
    │  database_dw.py  ETL
    ▼
data/database/sleep_dw.sqlite     esquema estrella
    │  queries.py / plots.py
    ▼
Notebook (EDA)  +  Streamlit (src/app.py)
```

| Módulo | Responsabilidad |
|---|---|
| `config.py` | Rutas relativas, selección de sujetos, parámetros metodológicos y umbrales |
| `extract.py` | Catálogo, selección, descarga validada y manifest |
| `bids.py` | Conversión a BIDS y metadatos |
| `preprocessing.py` | Lectura EDF, validación, epoching, recorte de vigilia, filtrado |
| `features.py` | Métricas espectrales y de complejidad |
| `quality.py` | Artefactos, outliers estadísticos, reporte de faltantes |
| `outliers.py` | Diagnóstico exploratorio de outliers: reglas univariadas, detectores multivariados y sensibilidad |
| `pipeline.py` | Orquestación del Transform con caché por registro |
| `database_oltp.py` / `database_dw.py` | Bases SQLite transaccional y analítica |
| `queries.py` / `plots.py` | Consultas SQL y figuras (compartidas con el dashboard) |
| `validation.py` | Checks automáticos con mensajes comprensibles |
| `app.py` | Dashboard Streamlit |

## Configuración

La primera celda prepara el entorno. En **Google Colab** clona el repositorio e instala las dependencias; localmente
solo agrega la raíz del proyecto al `sys.path`. Todas las rutas se calculan relativas a la raíz con `pathlib`.

`QUICK_RUN = True` procesa 2 sujetos (≈5 min); `False` usa la configuración completa (10 sujetos × 2 noches,
≈20–30 min la primera vez; las ejecuciones siguientes reutilizan el caché).

In [1]:
import os, sys, subprocess
from pathlib import Path

QUICK_RUN = False  # True: 2 sujetos para una prueba rápida

# TODO: reemplazar por la URL del repositorio en GitHub antes de ejecutar en Colab.
REPO_URL = "https://github.com/<usuario>/<repositorio>.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo_dir = Path("/content") / Path(REPO_URL).stem
    if not repo_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
    os.chdir(repo_dir)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    PROJECT_ROOT = repo_dir
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Raíz del proyecto:", PROJECT_ROOT)

Raíz del proyecto: C:\workspace\IID_TPF


In [2]:
import json, logging, random, warnings
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

from src import (config, extract, bids, preprocessing, features, quality, pipeline,
                 database_oltp, database_dw, queries, plots, validation)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
logging.getLogger("mne").setLevel(logging.WARNING)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

random.seed(config.RANDOM_SEED)
np.random.seed(config.RANDOM_SEED)

# Selección de sujetos: lista explícita de config (o los primeros N si no hay lista).
if config.SUBJECT_IDS:
    SELECTION = {"subject_ids": config.SUBJECT_IDS[:2] if QUICK_RUN else config.SUBJECT_IDS}
else:
    SELECTION = {"n_subjects": 2 if QUICK_RUN else config.N_SUBJECTS}

def show_checks(lines):
    """Muestra el resultado de un bloque de validaciones (cada una lanza ValidationError si falla)."""
    for line in lines:
        print("  ✔", line)

print("Selección:", SELECTION, "| noches:", config.NIGHTS)
print("Canales EEG:", config.EEG_CHANNELS)
print("Epoch:", config.EPOCH_DURATION_S, "s | filtro:", config.L_FREQ_HZ, "-", config.H_FREQ_HZ, "Hz",
      "| vigilia conservada:", config.WAKE_PADDING_MIN, "min")
print("Bandas:", config.FREQ_BANDS)
print("Umbrales de artefacto: ≥", config.ARTIFACT_MIN_CLIPPED_SAMPLES, "muestras saturadas; SD <", config.ARTIFACT_MIN_STD_UV, "µV")
print("Versiones:", pipeline.library_versions())

Selección: {'subject_ids': [5, 14, 20, 22, 33, 38, 40, 50, 65, 76]} | noches: (1, 2)
Canales EEG: ['EEG Fpz-Cz', 'EEG Pz-Oz']
Epoch: 30.0 s | filtro: 0.5 - 30.0 Hz | vigilia conservada: 30 min
Bandas: {'delta': (0.5, 4.0), 'theta': (4.0, 8.0), 'alpha': (8.0, 12.0), 'sigma': (12.0, 16.0), 'beta': (16.0, 30.0)}
Umbrales de artefacto: ≥ 10 muestras saturadas; SD < 0.5 µV
Versiones: {'python': '3.13.11', 'numpy': '2.2.6', 'pandas': '2.2.3', 'scipy': '1.15.3', 'mne': '1.9.0', 'mne-bids': '0.20.0', 'antropy': '0.1.9', 'pyarrow': '25.0.1', 'plotly': '7.1.0', 'streamlit': '1.65.0', 'scikit-learn': '1.6.1'}


## Extract

```
PhysioNet
    ↓
EDF PSG + Hypnogram
    ↓
data/raw/
```

1. Se descarga `SHA256SUMS.txt`, la lista oficial de archivos del dataset con sus hashes.
2. Con ella se arma el **catálogo** de registros SC, emparejando PSG e hipnograma por (sujeto, noche).
3. Se seleccionan los sujetos de `config.SUBJECT_IDS` (o los primeros `N_SUBJECTS`) y las noches de `config.NIGHTS`.
4. Cada archivo se descarga **solo si no existe o no es válido**; la validación exige tamaño mínimo y que el SHA256
   coincida con el oficial. La descarga es atómica (`.part` + renombrado), por lo que una interrupción no deja
   archivos corruptos.
5. Se registra todo en `data/raw/download_manifest.csv` (archivo, tamaño, hash, estado, URL de origen).

**Fuente**: se usa primero el *mirror* oficial de PhysioNet en AWS Open Data (~50× más rápido en las pruebas) y, si
falla, `physionet.org`. Como cada archivo se verifica contra los hashes oficiales, la fuente no altera los datos.

In [3]:
catalog = extract.build_catalog(extract.fetch_checksums())
print(f"Catálogo sleep-cassette: {len(catalog)} registros de {catalog['subject'].nunique()} sujetos")
print("Sujetos con una sola noche:", sorted(catalog.groupby("subject")["night"].count().loc[lambda s: s < 2].index))

manifest = extract.run_extract(**SELECTION)
display(manifest[["recording_id", "kind", "file", "size_bytes", "status"]])
show_checks(validation.validate_downloads(manifest))

INFO src.extract: Registros seleccionados: 20 (10 sujetos)


INFO src.extract: PSG       SC4051E0-PSG.edf         cached


INFO src.extract: Hypnogram SC4051EC-Hypnogram.edf   cached


INFO src.extract: PSG       SC4052E0-PSG.edf         cached


Catálogo sleep-cassette: 153 registros de 78 sujetos
Sujetos con una sola noche: [13, 36, 52]


INFO src.extract: Hypnogram SC4052EC-Hypnogram.edf   cached


... (34 outputs left out)

INFO src.extract: PSG       SC4762E0-PSG.edf         cached


INFO src.extract: Hypnogram SC4762EG-Hypnogram.edf   cached


,recording_id,kind,file,size_bytes,status
0,SC4051,PSG,data/raw/sleep-cassette/SC4051E0-PSG.edf,49651328,cached
1,SC4051,Hypnogram,data/raw/sleep-cassette/SC4051EC-Hypnogram.edf,3976,cached
2,SC4052,PSG,data/raw/sleep-cassette/SC4052E0-PSG.edf,51183488,cached
3,SC4052,Hypnogram,data/raw/sleep-cassette/SC4052EC-Hypnogram.edf,4232,cached
4,SC4141,PSG,data/raw/sleep-cassette/SC4141E0-PSG.edf,50271488,cached
5,SC4141,Hypnogram,data/raw/sleep-cassette/SC4141EU-Hypnogram.edf,3774,cached
6,SC4142,PSG,data/raw/sleep-cassette/SC4142E0-PSG.edf,50599808,cached
7,SC4142,Hypnogram,data/raw/sleep-cassette/SC4142EU-Hypnogram.edf,3826,cached
8,SC4201,PSG,data/raw/sleep-cassette/SC4201E0-PSG.edf,51147008,cached
9,SC4201,Hypnogram,data/raw/sleep-cassette/SC4201EC-Hypnogram.edf,2030,cached


  ✔ 40 archivos presentes en data/raw
  ✔ 20 registros con PSG + Hypnogram


## Organización BIDS

### EDF y BIDS no son lo mismo

- **EDF/EDF+** (*European Data Format*) es un **formato de archivo**: define cómo se codifican en binario las señales
  de un registro (header con canales, unidades, rango físico/digital, frecuencia de muestreo, y luego los bloques de
  datos). EDF+ agrega anotaciones, que en Sleep-EDF contienen el hipnograma.
- **BIDS** (*Brain Imaging Data Structure*) es una **convención de organización** de un *dataset* completo: nombres de
  carpetas y archivos (`sub-XX/ses-N/eeg/...`), y metadatos legibles en TSV/JSON (`participants.tsv`,
  `channels.tsv`, `events.tsv`, `dataset_description.json`). BIDS no reemplaza a EDF: **lo contiene** (EEG-BIDS
  admite EDF como formato de datos).

### Por qué BIDS mejora la reproducibilidad y la interoperabilidad

- Cualquier herramienta compatible (MNE-BIDS, EEGLAB, validadores BIDS) sabe dónde está cada dato y qué significa,
  sin código ad hoc para los nombres de Sleep-EDF.
- Los metadatos (sujetos, canales, eventos) quedan en texto plano, versionables y legibles por humanos.
- Separa claramente datos crudos, derivados y descripción del dataset.

### Decisiones

- `data/raw/` **no se modifica**. MNE-BIDS copia el EDF a `data/bids/` sin reescribir la señal (se verifica byte a
  byte durante el desarrollo).
- La sesión BIDS es la noche (`ses-1`, `ses-2`) y la tarea es `sleep`.
- Las etapas de sueño se exportan como eventos en `*_events.tsv`.
- `participants.tsv` se completa con edad y sexo de `SC-subjects.xls`, porque el EDF de Sleep-EDF no los incluye.
- `channels.tsv` se corrige con los valores del header EDF. MNE remuestrea internamente todos los canales a 100 Hz,
  pero el archivo real tiene EMG, respiración y temperatura a 1 Hz; los metadatos deben describir el archivo, no la
  representación en memoria.

In [4]:
bids_index = bids.run_bids(manifest)
display(bids_index)

print("Estructura (primer sujeto):")
first = sorted(config.BIDS_DIR.glob("sub-*"))[0]
for path in sorted(first.rglob("*")):
    if path.is_file():
        print("  ", path.relative_to(config.BIDS_DIR).as_posix())

display(pd.read_csv(config.BIDS_DIR / "participants.tsv", sep="\t"))
channels_tsv = next(first.rglob("*_channels.tsv"))
display(pd.read_csv(channels_tsv, sep="\t")[["name", "type", "units", "low_cutoff", "high_cutoff", "sampling_frequency"]])

INFO src.bids: BIDS existente, se reutiliza: sub-05_ses-1_task-sleep_eeg.edf


INFO src.bids: BIDS existente, se reutiliza: sub-05_ses-2_task-sleep_eeg.edf


INFO src.bids: BIDS existente, se reutiliza: sub-14_ses-1_task-sleep_eeg.edf


INFO src.bids: BIDS existente, se reutiliza: sub-14_ses-2_task-sleep_eeg.edf


INFO src.bids: BIDS existente, se reutiliza: sub-20_ses-1_task-sleep_eeg.edf


INFO src.bids: BIDS existente, se reutiliza: sub-20_ses-2_task-sleep_eeg.edf


... (14 outputs left out)

,recording_id,subject,night,bids_file
0,SC4051,5,1,data/bids/sub-05/ses-1/eeg/sub-05_ses-1_task-s...
1,SC4052,5,2,data/bids/sub-05/ses-2/eeg/sub-05_ses-2_task-s...
2,SC4141,14,1,data/bids/sub-14/ses-1/eeg/sub-14_ses-1_task-s...
3,SC4142,14,2,data/bids/sub-14/ses-2/eeg/sub-14_ses-2_task-s...
4,SC4201,20,1,data/bids/sub-20/ses-1/eeg/sub-20_ses-1_task-s...
5,SC4202,20,2,data/bids/sub-20/ses-2/eeg/sub-20_ses-2_task-s...
6,SC4221,22,1,data/bids/sub-22/ses-1/eeg/sub-22_ses-1_task-s...
7,SC4222,22,2,data/bids/sub-22/ses-2/eeg/sub-22_ses-2_task-s...
8,SC4331,33,1,data/bids/sub-33/ses-1/eeg/sub-33_ses-1_task-s...
9,SC4332,33,2,data/bids/sub-33/ses-2/eeg/sub-33_ses-2_task-s...


Estructura (primer sujeto):
   sub-05/ses-1/eeg/sub-05_ses-1_task-sleep_channels.tsv
   sub-05/ses-1/eeg/sub-05_ses-1_task-sleep_eeg.edf
   sub-05/ses-1/eeg/sub-05_ses-1_task-sleep_eeg.json
   sub-05/ses-1/eeg/sub-05_ses-1_task-sleep_events.json
   sub-05/ses-1/eeg/sub-05_ses-1_task-sleep_events.tsv
   sub-05/ses-1/sub-05_ses-1_scans.tsv
   sub-05/ses-2/eeg/sub-05_ses-2_task-sleep_channels.tsv
   sub-05/ses-2/eeg/sub-05_ses-2_task-sleep_eeg.edf
   sub-05/ses-2/eeg/sub-05_ses-2_task-sleep_eeg.json
   sub-05/ses-2/eeg/sub-05_ses-2_task-sleep_events.json
   sub-05/ses-2/eeg/sub-05_ses-2_task-sleep_events.tsv
   sub-05/ses-2/sub-05_ses-2_scans.tsv


,participant_id,age,sex
0,sub-05,28,F
1,sub-14,27,M
2,sub-20,51,F
3,sub-22,56,F
4,sub-33,60,M
5,sub-38,51,M
6,sub-40,67,F
7,sub-50,71,M
8,sub-65,88,F
9,sub-76,90,M


,name,type,units,low_cutoff,high_cutoff,sampling_frequency
0,EEG Fpz-Cz,EEG,µV,0.50,100.0,100.0
1,EEG Pz-Oz,EEG,µV,0.50,100.0,100.0
2,EOG horizontal,EOG,µV,0.50,100.0,100.0
3,Resp oro-nasal,RESP,NaN,0.03,0.9,1.0
4,EMG submental,EMG,µV,16.00,0.7,1.0
5,Temp rectal,TEMP,oC,NaN,NaN,1.0
6,Event marker,MISC,NaN,NaN,NaN,1.0


## Exploración inicial

Antes de procesar se inspecciona **cada** registro sin suponer valores: canales, frecuencia de muestreo y unidades
(leídas del header EDF), duración, anotaciones y coherencia temporal entre PSG e hipnograma. `validate_recording`
detiene el pipeline con un mensaje claro si falta un canal, la frecuencia o la unidad no son las esperadas, el PSG y
el hipnograma no comparten inicio, o aparece una etiqueta de etapa desconocida.

In [5]:
files = manifest.pivot(index=["recording_id", "subject", "night"], columns="kind", values="file").reset_index()
rows, warnings_found = [], []
for rec in files.itertuples(index=False):
    info = preprocessing.inspect_recording(config.PROJECT_ROOT / rec.PSG, config.PROJECT_ROOT / rec.Hypnogram)
    warnings_found += preprocessing.validate_recording(info)
    rows.append({
        "recording_id": rec.recording_id,
        "inicio": info["start_datetime"],
        "duración PSG [h]": round(info["psg_duration_s"] / 3600, 2),
        "fin hipnograma [h]": round(info["hypnogram_end_s"] / 3600, 2),
        "fs EEG [Hz]": sorted(set(info["eeg_sfreq_hz"].values())),
        "unidades EEG": sorted(set(info["eeg_units"].values())),
        "etiquetas": ", ".join(f"{k.replace('Sleep stage ', '')}:{v}" for k, v in sorted(info["annotation_counts"].items())),
    })
display(pd.DataFrame(rows))
print("Canales del PSG:", info["channels"])
print("Prefiltro de adquisición EEG:", info["eeg_prefilter"])
print(f"Advertencias ({len(warnings_found)}):")
for w in warnings_found:
    print("  -", w)

,recording_id,inicio,duración PSG [h],fin hipnograma [h],fs EEG [Hz],unidades EEG,etiquetas
0,SC4051,1989-04-26 16:25:00,22.68,24.0,[100.0],[uV],"1:19, 2:45, 3:39, 4:11, ?:1, R:3, W:12"
1,SC4052,1989-04-27 15:42:00,23.38,24.0,[100.0],[uV],"Movement time:2, 1:43, 2:32, 3:14, 4:6, ?:1, R..."
2,SC4141,1989-06-19 16:05:00,22.97,24.0,[100.0],[uV],"1:16, 2:30, 3:30, 4:18, ?:1, R:7, W:20"
3,SC4142,1989-06-20 15:33:00,23.12,24.0,[100.0],[uV],"1:18, 2:30, 3:29, 4:21, ?:1, R:8, W:17"
4,SC4201,1989-05-08 16:18:00,23.37,24.0,[100.0],[uV],"Movement time:1, 1:15, 2:15, 3:3, ?:1, R:6, W:15"
5,SC4202,1989-05-09 16:09:00,22.25,24.0,[100.0],[uV],"1:18, 2:11, ?:1, R:5, W:12"
6,SC4221,1989-05-24 16:55:00,22.50,24.0,[100.0],[uV],"1:45, 2:54, 3:29, 4:2, ?:1, R:9, W:24"
7,SC4222,1989-05-25 16:30:00,23.00,24.0,[100.0],[uV],"1:31, 2:54, 3:39, 4:19, ?:1, R:9, W:29"
8,SC4331,1990-03-19 15:15:00,23.42,24.0,[100.0],[uV],"Movement time:1, 1:48, 2:27, ?:1, R:9, W:36"
9,SC4332,1990-03-20 15:20:00,23.00,24.0,[100.0],[uV],"Movement time:2, 1:44, 2:27, ?:1, R:9, W:32"


Canales del PSG: ['EEG Fpz-Cz', 'EEG Pz-Oz', 'EOG horizontal', 'Resp oro-nasal', 'EMG submental', 'Temp rectal', 'Event marker']
Prefiltro de adquisición EEG: {'EEG Fpz-Cz': 'HP:0.5Hz LP:100Hz [enhanced cassette BW]', 'EEG Pz-Oz': 'HP:0.5Hz LP:100Hz [enhanced cassette BW]'}
Advertencias (20):
  - SC4051E0-PSG.edf: el hipnograma termina en 86400 s y la señal en 81660 s; los epochs sin señal se descartan al segmentar
  - SC4052E0-PSG.edf: el hipnograma termina en 86400 s y la señal en 84180 s; los epochs sin señal se descartan al segmentar
  - SC4141E0-PSG.edf: el hipnograma termina en 86400 s y la señal en 82680 s; los epochs sin señal se descartan al segmentar
  - SC4142E0-PSG.edf: el hipnograma termina en 86400 s y la señal en 83220 s; los epochs sin señal se descartan al segmentar
  - SC4201E0-PSG.edf: el hipnograma termina en 86400 s y la señal en 84120 s; los epochs sin señal se descartan al segmentar
  - SC4202E0-PSG.edf: el hipnograma termina en 86400 s y la señal en 80100 s; los

Observaciones:

- Todos los registros tienen los dos canales EEG a 100 Hz, en µV, con prefiltro de hardware 0.5–100 Hz.
- Los conteos de etiquetas son **segmentos** del hipnograma, no epochs (un segmento W puede durar horas).
- En todos los registros el hipnograma se extiende más allá del final de la señal: esos epochs se descartan y se
  contabilizan como faltantes (ver *Tratamiento de datos faltantes*).

## Transform

Cada registro se procesa **secuencialmente**, para no tener varias señales en memoria:

1. **Epochs**: los segmentos del hipnograma se dividen en epochs de 30 s. Se verifica que las anotaciones sean
   contiguas y estén alineadas a 30 s; si hubiera huecos, el pipeline se detiene.
2. **Recorte al período de sueño nocturno**: los registros SC duran ~20 h y la mayor parte es vigilia diurna. Se
   conservan **30 min de W antes del primer epoch de sueño y 30 min después del último** (práctica habitual con este
   dataset). Sin recorte, W representaría ~70 % de los epochs y dominaría cualquier comparación.
   - El primer y el último epoch de sueño se buscan solo **dentro de la franja 21:00–12:00** (hora local del
     registro, `NIGHT_START_HOUR` / `NIGHT_END_HOUR`). Sin esta restricción, las **siestas diurnas** de los sujetos
     mayores (por ejemplo, a las 13:00 o a las 17:00) estiraban la "noche" a casi 24 h.
   - La franja se eligió después de comparar alternativas. Separar bloques de sueño por huecos de vigilia partía las
     noches con despertares largos, que son frecuentes en las personas mayores.
   - Los epochs recortados no se eliminan de la tabla de epochs: quedan marcados (`in_analysis_window = False`).
3. **Carga selectiva**: solo se leen los 2 canales EEG. Esto evita que MNE suba a 100 Hz los canales de 1 Hz y reduce
   la memoria.
4. **Filtro pasa-banda 0.5–30 Hz** (FIR de fase cero, el valor por defecto de MNE), aplicado a la señal **continua**
   recortada, con 30 s de margen a cada lado, antes de segmentar. Filtrar cada epoch por separado introduciría
   efectos de borde.
   - El pasa-altos de 0.5 Hz elimina la deriva lenta. El hardware ya aplica 0.5 Hz, pero así se garantiza el mismo
     corte en todos los registros.
   - El pasa-bajos de 30 Hz coincide con el límite superior de la banda beta analizada y atenúa actividad muscular y
     ruido de red (50 Hz, cerca de Nyquist).
   - No se aplica ninguna otra transformación (ni re-referencia, ni ICA, ni normalización): para las métricas
     elegidas no son necesarias y agregarían supuestos.
5. **Array de epochs** en µV, float32 (precisión suficiente para la señal y la mitad de memoria). Las features se
   calculan en float64, porque acumular sumas espectrales en float32 introduce errores de ~1e-6.
6. **Features y flags de artefacto**, guardados como Parquet por registro (caché).

El caché se invalida automáticamente si cambian los parámetros metodológicos de `config.py` o la versión del código
del Transform (se guarda un hash de ambos).

A continuación se ilustra el efecto del filtro sobre un epoch y luego se ejecuta el pipeline completo.

In [6]:
import plotly.graph_objects as go

rec = files.iloc[0]
psg, hyp = config.PROJECT_ROOT / rec.PSG, config.PROJECT_ROOT / rec.Hypnogram
info = preprocessing.inspect_recording(psg, hyp)
ep = preprocessing.mark_analysis_window(
    preprocessing.annotations_to_epochs(preprocessing.read_hypnogram(hyp), info["psg_duration_s"]),
    info["start_datetime"])
example = ep[ep["in_analysis_window"] & ep["has_signal"] & (ep["stage"] == "N2")].iloc[20]

raw_unfiltered = preprocessing.load_eeg(psg, hyp)
raw_filtered = preprocessing.crop_and_filter(raw_unfiltered.copy(), ep)
x_raw = preprocessing.extract_epoch_array(raw_unfiltered, np.array([example.start_sec]))[0, 0]
x_filt = preprocessing.extract_epoch_array(raw_filtered, np.array([example.start_sec]))[0, 0]
del raw_unfiltered, raw_filtered

t = np.arange(x_raw.size) / config.EXPECTED_SFREQ_HZ
fig = go.Figure()
fig.add_trace(go.Scatter(x=t, y=x_raw, name="Original", line=dict(color=plots.INK_MUTED, width=1)))
fig.add_trace(go.Scatter(x=t, y=x_filt, name="Filtrado 0.5–30 Hz", line=dict(color=plots.STAGE_COLORS["N2"], width=1)))
fig.update_xaxes(title="Tiempo dentro del epoch [s]")
fig.update_yaxes(title="Amplitud [µV]")
plots._style(fig, f"Fpz-Cz · {rec.recording_id} · epoch {example.epoch_number} (N2)", height=360).show()

In [7]:
results = pipeline.run_pipeline(manifest)
tidy, epochs, recordings, missing = results["tidy"], results["epochs"], results["recordings"], results["missing"]
print(f"Tabla tidy: {tidy.shape[0]:,} filas × {tidy.shape[1]} columnas, "
      f"{tidy.memory_usage(deep=True).sum() / 1e6:.1f} MB en memoria")

INFO src.pipeline: SC4051: caché reutilizado


INFO src.pipeline: SC4052: caché reutilizado


INFO src.pipeline: SC4141: caché reutilizado


INFO src.pipeline: SC4142: caché reutilizado


INFO src.pipeline: SC4201: caché reutilizado


INFO src.pipeline: SC4202: caché reutilizado


... (12 outputs left out)

INFO src.pipeline: SC4761: caché reutilizado


INFO src.pipeline: SC4762: caché reutilizado


INFO src.pipeline: Tidy: 46212 filas x 27 columnas (15.1 MB en memoria)


Tabla tidy: 46,212 filas × 27 columnas, 15.1 MB en memoria


## Hipnograma y epochs

Las etiquetas originales (R&K) se mapean a la nomenclatura AASM:

| Hipnograma | Etapa | Tratamiento |
|---|---|---|
| `Sleep stage W` | W | Puntuada |
| `Sleep stage 1` | N1 | Puntuada |
| `Sleep stage 2` | N2 | Puntuada |
| `Sleep stage 3`, `Sleep stage 4` | N3 | Puntuada (R&K S3 + S4 = AASM N3) |
| `Sleep stage R` | REM | Puntuada |
| `Movement time` | MOVE | **No es una etapa**: se conserva, no se imputa |
| `Sleep stage ?` | UNK | **Desconocida**: se conserva, no se imputa |

MOVE y `?` no se reasignan a una etapa vecina: hacerlo inventaría información que el experto no registró. Se
conservan con su propia categoría, figuran en las bases y se excluyen de las comparaciones entre etapas.

In [8]:
display(recordings[["recording_id", "subject", "night", "age", "sex", "lights_off",
                    "n_epochs_hypnogram", "n_epochs_analyzed"]])
counts = (epochs.assign(grupo=np.select(
              [~epochs["has_signal"], ~epochs["in_analysis_window"]],
              ["sin señal", "recortado (vigilia)"], "analizado"))
          .pivot_table(index="stage", columns="grupo", values="epoch_number", aggfunc="count", fill_value=0))
display(counts)

,recording_id,subject,night,age,sex,lights_off,n_epochs_hypnogram,n_epochs_analyzed
0,SC4051,5,1,28,F,01:22:00,2880,672
1,SC4052,5,2,28,F,00:35:00,2880,1248
2,SC4141,14,1,27,M,22:55:00,2880,1004
3,SC4142,14,2,27,M,23:22:00,2880,952
4,SC4201,20,1,51,F,23:10:00,2880,1023
5,SC4202,20,2,51,F,23:15:00,2880,1021
6,SC4221,22,1,56,F,23:47:00,2880,1099
7,SC4222,22,2,56,F,23:14:00,2880,1108
8,SC4331,33,1,60,M,22:58:00,2880,988
9,SC4332,33,2,60,M,22:55:00,2880,1059


grupo,analizado,recortado (vigilia),sin señal
stage,,,
MOVE,22,0,0
N1,2161,139,0
N2,9254,265,0
N3,1317,24,0
REM,3662,0,0
UNK,154,0,2438
W,6536,31628,0


In [9]:
obs_preview = tidy.rename(columns={"sleep_stage": "stage_code", "start_time_sec": "start_sec"})
obs_preview["channel"] = obs_preview["channel"].astype(str).str.replace("EEG ", "")
onset = (obs_preview[obs_preview["stage_code"].isin(["N1", "N2", "N3", "REM"])]
         .groupby("recording_id")["start_sec"].min())
obs_preview["hours_since_sleep_onset"] = (obs_preview["start_sec"] - obs_preview["recording_id"].map(onset)) / 3600
obs_preview["stage_code"] = obs_preview["stage_code"].astype(str)
plots.hypnogram(obs_preview, files.iloc[0].recording_id).show()

## Control de calidad

Además de las validaciones de entrada (sección *Exploración inicial*), sobre la tabla resultante se verifica que:

- no haya claves (registro, epoch, canal) duplicadas;
- cada epoch tenga todos los canales;
- todos los epochs empiecen en múltiplos de 30 s;
- las etapas sean válidas;
- la potencia relativa esté en [0, 1] y sume 1;
- `is_artifact` sea coherente con `artifact_reason`;
- no haya features NaN en filas sin artefacto.

In [10]:
show_checks(validation.validate_tidy(tidy))

  ✔ Sin claves (recording, epoch, channel) duplicadas
  ✔ Canales: ['EEG Fpz-Cz', 'EEG Pz-Oz']
  ✔ Cada epoch tiene todos los canales
  ✔ Inicios de epoch múltiplos de 30 s
  ✔ Etapas válidas: ['MOVE', 'N1', 'N2', 'N3', 'REM', 'UNK', 'W']
  ✔ Potencia relativa en [0, 1] y suma 1 ± 1e-09 (sin artefactos)
  ✔ is_artifact coherente con artifact_reason
  ✔ Sin features NaN en filas no artefactadas


## Tratamiento de datos faltantes

Se distinguen cuatro situaciones. Ninguna se resuelve por imputación:

1. **Epochs del hipnograma sin señal**: el hipnograma continúa después del final del PSG. Sin señal no hay
   features: se excluyen y se cuentan.
2. **Vigilia fuera de la ventana de análisis**: no es un faltante, sino una **exclusión por diseño** (recorte de
   vigilia). Se reporta para que el volumen de datos descartado sea explícito.
3. **Etapa desconocida (`?`) o Movement time dentro de la ventana**: la señal existe y se calculan sus features, pero
   la etapa es desconocida. Se conservan como UNK/MOVE y se excluyen de las comparaciones entre etapas.
   Transformarlas en una etapa (por ejemplo, copiando la del epoch anterior) sería fabricar una etiqueta.
4. **Features no calculables**: si la señal es plana o tiene valores no finitos, la Sample Entropy, la Spectral
   Entropy y las potencias relativas no están definidas. Se guardan como NaN (`NULL` en SQL) y no como 0, porque
   `antropy` devolvería 0, que se leería como "regularidad perfecta".

In [11]:
display(missing.style.format({"percent": "{:.2f} %"}).hide(axis="index"))

level,item,count,denominator,percent,cause,decision
epoch,Epoch del hipnograma sin señal EEG,2438,57600,4.23 %,El hipnograma se extiende más allá del final del PSG,Excluido: no hay señal de la cual extraer features
epoch,Fuera de la ventana nocturna (vigilia y siestas),32056,57600,55.65 %,Registros de ~20 h; se conservan 30 min de W antes/después del sueño nocturno (21:00-12:00),Excluido por diseño (no es un faltante): evita que W y las siestas diurnas dominen el análisis
epoch,Etapa desconocida ('?') en ventana,154,57600,0.27 %,El experto no asignó etapa,Se conserva con etapa UNK; no se imputa; se excluye de comparaciones entre etapas
epoch,Movement time en ventana,22,57600,0.04 %,Movimiento corporal: el experto no pudo asignar etapa,Se conserva con etapa MOVE; no se imputa; se excluye de comparaciones entre etapas
epoch,Etapa sin mapeo,0,57600,0.00 %,Descripción del hipnograma no contemplada en STAGE_MAPPING,Debe ser 0 (la validación de cada registro lo impide)
feature,delta_abs = NaN,0,46212,0.00 %,"No calculable (señal plana o no finita, o SampEn sin coincidencias)",Se deja NaN (no se imputa); la fila conserva su flag de artefacto
feature,theta_abs = NaN,0,46212,0.00 %,"No calculable (señal plana o no finita, o SampEn sin coincidencias)",Se deja NaN (no se imputa); la fila conserva su flag de artefacto
feature,alpha_abs = NaN,0,46212,0.00 %,"No calculable (señal plana o no finita, o SampEn sin coincidencias)",Se deja NaN (no se imputa); la fila conserva su flag de artefacto
feature,sigma_abs = NaN,0,46212,0.00 %,"No calculable (señal plana o no finita, o SampEn sin coincidencias)",Se deja NaN (no se imputa); la fila conserva su flag de artefacto
feature,beta_abs = NaN,0,46212,0.00 %,"No calculable (señal plana o no finita, o SampEn sin coincidencias)",Se deja NaN (no se imputa); la fila conserva su flag de artefacto


### Perfil de nulos y ceros

Antes de decidir un tratamiento hay que distinguir **qué significa** cada nulo y cada cero. Un valor faltante no
siempre es un dato perdido, y un cero puede ser un valor válido, un faltante disfrazado o un sensor caído.

In [12]:
feature_columns = features.FEATURE_COLUMNS
display(quality.data_profile(tidy[feature_columns]).round(3))
print("Nulos en columnas de texto de la tabla tidy:",
      tidy.select_dtypes(exclude="number").isna().sum().loc[lambda s: s > 0].to_dict())
print("Nulos en recordings:", recordings.isna().sum().loc[lambda s: s > 0].to_dict() or "ninguno")
print("Ceros en epochs:", quality.data_profile(epochs).query("ceros > 0")["ceros"].to_dict())

,nulos,infinitos,ceros,negativos,mínimo,mediana,máximo
delta_abs,0,0,0,0,0.794,52.728,5787.541
theta_abs,0,0,0,0,0.036,9.779,467.152
alpha_abs,0,0,0,0,0.007,4.650,191.058
sigma_abs,0,0,0,0,0.002,2.209,152.353
beta_abs,0,0,0,0,0.017,2.015,805.113
delta_rel,0,0,0,0,0.047,0.744,0.995
theta_rel,0,0,0,0,0.002,0.111,0.625
alpha_rel,0,0,0,0,0.001,0.052,0.823
sigma_rel,0,0,0,0,0.000,0.026,0.241
beta_rel,0,0,0,0,0.000,0.026,0.611


Nulos en columnas de texto de la tabla tidy: {'artifact_reason': 46122, 'outlier_features': 41521}
Nulos en recordings: ninguno
Ceros en epochs: {'epoch_number': 20, 'start_sec': 20}


In [13]:
# Valores casi nulos: el cero exacto no aparece, pero sí valores muy bajos.
low_amplitude = tidy[tidy["rms"] < 2]
print(f"RMS < 2 µV: {len(low_amplitude):,} filas · por registro:",
      low_amplitude["recording_id"].value_counts().head(4).to_dict())
display(tidy.groupby(["recording_id", "channel"], observed=True)["rms"].median().unstack().round(1)
        .sort_values("EEG Pz-Oz").head(3))
columns = ["recording_id", "epoch_number", "channel", "sleep_stage", "sample_entropy", "rms", "peak_to_peak"]
display(tidy.loc[tidy["sample_entropy"] < 0.05, columns].round(3))

RMS < 2 µV: 1,223 filas · por registro: {'SC4221': 750, 'SC4762': 298, 'SC4761': 98, 'SC4501': 77}


channel,EEG Fpz-Cz,EEG Pz-Oz
recording_id,,
SC4221,15.5,1.9
SC4381,8.0,4.5
SC4331,8.6,6.0


,recording_id,epoch_number,channel,sleep_stage,sample_entropy,rms,peak_to_peak
39944,SC4761,528,EEG Fpz-Cz,W,0.037,20.299,247.366
39992,SC4761,552,EEG Fpz-Cz,W,0.037,21.428,259.533
43031,SC4761,2071,EEG Pz-Oz,W,0.045,29.599,255.272
43975,SC4762,1266,EEG Pz-Oz,UNK,0.046,26.158,190.774
45224,SC4762,1891,EEG Fpz-Cz,W,0.030,20.551,186.792
45225,SC4762,1891,EEG Pz-Oz,W,0.041,18.963,168.903
45254,SC4762,1906,EEG Fpz-Cz,UNK,0.009,18.682,175.205
45255,SC4762,1906,EEG Pz-Oz,UNK,0.033,13.175,153.206
45266,SC4762,1912,EEG Fpz-Cz,UNK,0.032,26.815,191.900
45267,SC4762,1912,EEG Pz-Oz,UNK,0.030,29.441,229.232


**Lectura de los perfiles** (valores de esta ejecución):

- **Nulos en features: 0.** La regla de "NaN si no es calculable" existe, pero no se activó en esta muestra.
- **Nulos estructurales:** `artifact_reason` es nulo en el 99.8 % de las filas y `outlier_features` en el 90 %. No son
  datos perdidos: significan "sin artefacto" y "sin outlier". **No se imputan.**
- **Ceros exactos: 0 en las 15 features.** Es lo esperable: una potencia o una entropía exactamente en 0 solo
  aparecería con señal plana. Los únicos ceros están en `epoch_number` y `start_sec` de los 20 registros, y son el
  primer epoch, un valor válido.
- **Consecuencia práctica:** a diferencia del dataset de clase, ninguna variable tiene Q1 = 0, así que el IQR no se
  anula. El problema de la clase con `RobustScaler` no aparece.
- **Ceros en la señal:** el cero exacto no sirve como criterio, porque la conversión digital del EDF tiene offset y
  ningún valor cae en 0 µV. Lo que importa es la señal **constante**, y la racha constante más larga es de 48 muestras
  (0.48 s), en un tramo saturado (ver la subsección *Saturación*).
- **Casi ceros:** el criterio `flat_signal` (SD < 0.5 µV) no se activó y el RMS mínimo es 0.99 µV. Las 1 223 filas con
  RMS < 2 µV se concentran en SC4221 (750): su canal Pz-Oz tiene una mediana de 1.9 µV, frente a 7.65 µV del conjunto.
  Es una **diferencia del registro**, no de epochs aislados.
- **Sample Entropy casi nula:** 56 filas < 0.05, todas en SC4761/SC4762. 50 son epochs `?` de SC4762, que el experto
  no pudo puntuar, y 6 son W. Tienen RMS de 10–42 µV y pico a pico de 127–260 µV, así que no son señal plana. Es
  compatible con un transitorio breve: infla el desvío estándar, y con él la tolerancia `r = 0.2·SD`, de modo que casi
  todos los patrones coinciden. Son valores **extremos de cálculo**, no evidencia de "regularidad". Los epochs `?` ya
  quedan fuera de las comparaciones entre etapas.

## Tratamiento de outliers y artefactos

### Outlier estadístico ≠ artefacto de señal

| | Outlier estadístico | Artefacto de señal |
|---|---|---|
| Definición | Valor extremo respecto de la distribución observada | El registro no refleja actividad cerebral |
| Criterio | Regla de Tukey: fuera de [Q1 − 1.5·IQR, Q3 + 1.5·IQR] | Criterios físicos sobre la señal |
| Ejemplo | Una onda lenta de gran amplitud en N3 | Electrodo desconectado, saturación, movimiento |
| Decisión | **Solo se marca** (`is_stat_outlier`, `outlier_features`) | **Se marca** (`is_artifact`, `artifact_reason`) y se excluye por defecto de los análisis |

**Ninguna fila se elimina.** Un valor extremo puede ser fisiología real, y eliminarlo sesgaría justamente las etapas
más interesantes.

### Criterios de artefacto (configurables en `config.py`)

- `non_finite`: algún valor NaN/inf en el epoch.
- `flat_signal`: desvío estándar < `ARTIFACT_MIN_STD_UV` (0.5 µV), compatible con un electrodo desconectado.
- `clipped`: al menos `ARTIFACT_MIN_CLIPPED_SAMPLES` muestras (10, es decir 0.1 s) en el límite físico del EDF, contadas
  sobre la señal sin filtrar. Indica que el conversor se saturó y que la señal real quedó recortada. Un umbral de
  amplitud no sirve aquí (ver *Saturación*): las ondas lentas grandes de N3 en Fpz-Cz llegan a ≈ 440 µV tras el filtro, y
  un umbral más bajo marcaría sueño profundo como artefacto.

### Outliers estadísticos

La regla de Tukey se aplica **por registro × canal × etapa**. Las distribuciones difieren legítimamente entre etapas
(N3 tiene más delta que REM por definición) y también entre noches, por edad o ganancia del equipo: cada epoch se
compara con su propia noche (ver *¿Epochs raros o registros distintos?*). Las potencias absolutas se evalúan en log10,
porque su distribución es muy asimétrica.

In [14]:
artifact_summary = (tidy.groupby(["recording_id", "channel"], observed=True)
                    .agg(observaciones=("is_artifact", "size"), artefactos=("is_artifact", "sum"),
                         outliers_estadisticos=("is_stat_outlier", "sum")))
artifact_summary["% outliers"] = (100 * artifact_summary["outliers_estadisticos"] / artifact_summary["observaciones"]).round(1)
display(artifact_summary)
print("Motivos de artefacto:", tidy["artifact_reason"].value_counts().to_dict() or "ninguno")
print("Features más frecuentes entre los outliers estadísticos:")
display(tidy["outlier_features"].str.split(";").explode().value_counts().to_frame("filas"))

observaciones  artefactos  outliers_estadisticos  % outliers
recording_id channel                                                                 
SC4051       EEG Fpz-Cz            672           2                     57         8.5
             EEG Pz-Oz             672           0                     96        14.3
SC4052       EEG Fpz-Cz           1248          15                    104         8.3
             EEG Pz-Oz            1248           0                    135        10.8
SC4141       EEG Fpz-Cz           1004           0                     99         9.9
             EEG Pz-Oz            1004           0                    100        10.0
SC4142       EEG Fpz-Cz            952           0                     98        10.3
             EEG Pz-Oz             952           0                    110        11.6
SC4201       EEG Fpz-Cz           1023          34                     83         8.1
             EEG Pz-Oz            1023           2                     87         8.5
SC4202       EEG Fpz-Cz           1021           0                     58         5.7
             EEG Pz-Oz            1021           0                     85         8.3
SC4221       EEG Fpz-Cz           1099           2                     88         8.0
             EEG Pz-Oz            1099           0                    115        10.5
SC4222       EEG Fpz-Cz           1108          12                     80         7.2
             EEG Pz-Oz            1108           0                    124        11.2
SC4331       EEG Fpz-Cz            988           1                     87         8.8
             EEG Pz-Oz             988           0                     87         8.8
SC4332       EEG Fpz-Cz           1059           3                     82         7.7
             EEG Pz-Oz            1059           0                    110        10.4
SC4381       EEG Fpz-Cz           1029           0                     89         8.6
             EEG Pz-Oz            1029           0                     94         9.1
SC4382       EEG Fpz-Cz           1034           0                     93         9.0
             EEG Pz-Oz            1034           2                    104        10.1
SC4401       EEG Fpz-Cz           1064           0                    107        10.1
             EEG Pz-Oz            1064           0                    145        13.6
SC4402       EEG Fpz-Cz           1072           0                     41         3.8
             EEG Pz-Oz            1072           0                    117        10.9
SC4501       EEG Fpz-Cz           1326           0                    119         9.0
             EEG Pz-Oz            1326           0                    264        19.9
SC4502       EEG Fpz-Cz           1103           0                     60         5.4
             EEG Pz-Oz            1103           0                    127        11.5
SC4651       EEG Fpz-Cz           1762           0                    181        10.3
             EEG Pz-Oz            1762          17                    223        12.7
SC4652       EEG Fpz-Cz           1363           0                     89         6.5
             EEG Pz-Oz            1363           0                    165        12.1
SC4761       EEG Fpz-Cz           1703           0                    175        10.3
             EEG Pz-Oz            1703           0                    228        13.4
SC4762       EEG Fpz-Cz           1476           0                    262        17.8
             EEG Pz-Oz            1476           0                    123         8.3

Motivos de artefacto: {'clipped': 90}
Features más frecuentes entre los outliers estadísticos:


,filas
outlier_features,
peak_to_peak,1687
alpha_rel,1651
rms,1643
total_power,1203
sample_entropy,1110
spectral_entropy,1013
delta_abs,1010
delta_rel,933


La figura siguiente responde: **¿dónde caen los valores extremos de amplitud?** Los puntos son los valores fuera de
los bigotes del box plot. Las amplitudes más altas aparecen en N3 y corresponden a ondas lentas, no a artefactos.

In [15]:
plots.amplitude_by_stage(obs_preview).show()

### Diagnóstico de outliers

Esta subsección aplica los métodos de la clase 05 y los adapta al EEG. Las preguntas son qué tan válidas son las
reglas usadas, si los métodos coinciden entre sí y cuánto cambiarían los resultados según cómo se traten los valores
extremos. Este análisis justificó dos decisiones del pipeline: marcar la **saturación** como artefacto y calcular
Tukey **dentro de cada noche**. Los detectores multivariados son exploratorios y no modifican ninguna tabla.

#### Saturación: por qué reemplazó al umbral de amplitud

Los EDF tienen un rango físico fijo por canal y registro (entre ±147 y ±248 µV en esta muestra). Una señal que supera
ese rango queda **recortada** en el límite. Por eso se cuentan las muestras en el límite físico, sobre la señal **sin
filtrar**, porque el filtro redondea las mesetas.

In [16]:
from src import outliers

is_clipped = tidy["artifact_reason"].eq("clipped")
print(f"Pico a pico máximo tras el filtro: {tidy['peak_to_peak'].max():.0f} µV")
print(f"Filas saturadas: {is_clipped.sum()} ({100 * is_clipped.mean():.2f} %)")
display(pd.crosstab(tidy["sleep_stage"].astype(str), tidy["channel"].astype(str),
                    values=is_clipped, aggfunc="sum").fillna(0).astype(int))
print("Por registro:", tidy.loc[is_clipped, "recording_id"].value_counts().head(5).to_dict())

Pico a pico máximo tras el filtro: 437 µV
Filas saturadas: 90 (0.19 %)


channel,EEG Fpz-Cz,EEG Pz-Oz
sleep_stage,,
MOVE,0,0
N1,1,0
N2,5,10
N3,0,0
REM,6,2
UNK,0,0
W,57,9


Por registro: {'SC4201': 36, 'SC4651': 17, 'SC4052': 15, 'SC4222': 12, 'SC4332': 3}


**Corrección del criterio.** Una versión anterior del pipeline usaba un umbral de pico a pico de 500 µV y daba "0
artefactos". Ese resultado no demostraba que los datos estuvieran limpios: con un rango físico máximo de ±248 µV, el
pico a pico sin filtrar no puede llegar a 496 µV, y el máximo observado era de 415 µV. La señal se recorta antes de
alcanzar el umbral, que era **prácticamente inalcanzable**. Por eso se reemplazó por el criterio de saturación.

- Hay **90 filas (0.19 %)** con al menos 10 muestras saturadas (0.1 s).
- El 73 % está en W (66), lo que es compatible con movimientos, y **ninguna en N3**: las ondas lentas grandes no se
  recortan, por lo que el criterio no elimina fisiología de sueño profundo.
- Estas filas conservan sus features y se excluyen por defecto de los análisis, igual que cualquier artefacto.

#### Reglas univariadas: ¿qué tan válido es el umbral?

Como se planteó en clase, `|z| > 3` supone normalidad. La tabla muestra la forma de cada distribución dentro de canal
× etapa (sin artefactos) y el porcentaje que marca cada regla. Bajo normalidad, `|z| > 3` marcaría 0.27 %.

In [17]:
display(outliers.compare_univariate_rules(tidy).round(2))

,asimetría,curtosis,A²,% |z| > 3,% Tukey 1.5,% Tukey 3
variable,,,,,,
delta_abs,-0.46,1.40,15.26,0.94,3.45,0.23
total_power,-0.82,2.23,40.27,1.59,4.63,0.97
delta_rel,-0.41,-0.13,16.85,0.27,0.54,0.02
alpha_rel,1.76,5.08,189.07,1.58,5.66,1.02
sample_entropy,0.35,1.58,4.66,0.97,1.87,0.32
spectral_entropy,-0.42,0.04,20.07,0.38,1.06,0.05
rms,1.62,8.74,65.40,1.13,2.98,0.87
peak_to_peak,1.38,5.57,60.40,1.16,3.24,0.71


- **Ninguna variable es normal** (A² entre 4.7 y 189, frente a un valor crítico de ≈ 1.1 al 1 %). Además, los epochs de
  una noche no son independientes, así que A² se usa como descripción y no como prueba formal.
- Las colas pesadas (`rms` con curtosis 8.7, `peak_to_peak` con 5.6) llevan a `|z| > 3` a marcar hasta 1.6 %, casi seis
  veces lo esperado. El umbral de 3 desvíos **no es un percentil real** con estos datos.
- Tukey no asume normalidad, por eso es la regla más razonable aquí. Pero 1.5·IQR marca hasta 5.7 %, y sobre 8
  variables se acumula.

#### ¿Epochs raros o registros distintos?

Una versión anterior comparaba cada epoch contra **todos los registros** de su canal y etapa, y marcaba 15.1 % de las
filas. La figura compara ese criterio con el actual, que usa como referencia **la misma noche**.

In [18]:
tukey_between, tukey_rates = outliers.tukey_rate_by_recording(tidy)
print(f"Filas marcadas: entre registros {100 * tukey_between.mean():.1f} % · "
      f"dentro del registro {100 * tidy['is_stat_outlier'].mean():.1f} %")
plots.outlier_rate_by_recording(tukey_rates).show()

Filas marcadas: entre registros 15.1 % · dentro del registro 10.2 %


Con la referencia global, SC4221 tiene 52 % de "outliers" y SC4502 35 %, mientras que SC4331 tiene 3 %. Esa
regla detecta **qué persona o qué ganancia tiene el registro**, no epochs extraños. Dentro de cada noche la tasa queda
entre 7 % y 14 % en todos los registros, con 10.2 % en total. El grupo de referencia cambia más el resultado que el
umbral, y por eso el pipeline usa la referencia dentro de la noche.

#### Detectores multivariados

Una fila puede ser normal en cada variable y rara en la combinación. Se aplican 5 detectores de la clase sobre 8
features (log de potencias y amplitud, y entropías), con z robusto por registro × canal, de modo que cada epoch se mida
contra su noche. Siguiendo la conclusión de la clase, todos marcan **la misma proporción de filas (1 %)**, y DBSCAN
queda afuera porque no da un puntaje continuo.

No hay una etiqueta de anomalía como `NSP`. Como referencias parciales se usan la **saturación** y los epochs
**Movement time** del experto.

In [19]:
scores = outliers.multivariate_scores(tidy)
flags = outliers.flag_top(scores)
votes = flags.sum(axis=1)
consensus = votes >= config.OUTLIER_MIN_VOTES

print("Índice de Jaccard entre las filas marcadas por cada detector:")
display(outliers.overlap(flags).round(2))
print("ROC-AUC contra las referencias parciales:")
display(outliers.reference_auc(scores, {"Saturación": is_clipped, "Movement time": tidy["sleep_stage"].eq("MOVE")}).round(3))
print("Votos por fila:", votes.value_counts().sort_index().to_dict())
print(f"Consenso (≥ {config.OUTLIER_MIN_VOTES} de 5): {consensus.sum()} filas ·",
      tidy.loc[consensus, "sleep_stage"].astype(str).value_counts().to_dict())

Índice de Jaccard entre las filas marcadas por cada detector:


,Distancia al centro,k-means,k-NN,LOF,Isolation Forest
Distancia al centro,1.00,0.52,0.17,0.03,0.59
k-means,0.52,1.00,0.28,0.04,0.33
k-NN,0.17,0.28,1.00,0.10,0.13
LOF,0.03,0.04,0.10,1.00,0.03
Isolation Forest,0.59,0.33,0.13,0.03,1.00


ROC-AUC contra las referencias parciales:


,Saturación,Movement time
Distancia al centro,0.973,0.593
k-means,0.936,0.617
k-NN,0.945,0.560
LOF,0.502,0.525
Isolation Forest,0.970,0.602


Votos por fila: {0: 44881, 1: 742, 2: 297, 3: 207, 4: 67, 5: 18}
Consenso (≥ 4 de 5): 85 filas · {'W': 59, 'N1': 12, 'N3': 9, 'N2': 2, 'MOVE': 1, 'REM': 1, 'UNK': 1}


- **Los detectores no coinciden mucho.** La superposición va de 0.03 a 0.59. Distancia al centro e Isolation Forest
  miden casi lo mismo (extremos globales) y LOF encuentra otra cosa (rarezas locales). No existe "el" conjunto de
  outliers: depende de la definición.
- **Saturación:** ROC-AUC de 0.94–0.97, salvo LOF (0.50). El resultado es en parte esperable, porque la amplitud está
  entre las features. Pero marcar el 1 % es exigente: solo 4 de las 90 filas saturadas llegan al consenso. Para
  encontrar saturación conviene el criterio físico directo, no un detector.
- **Movement time:** ROC-AUC de 0.53–0.62, apenas mejor que el azar. Los detectores **no** reemplazan al experto para
  identificar movimiento.
- **Consenso:** 85 filas (0.18 %), 59 de ellas en W y 22 en SC4221. Sin una etiqueta no se puede afirmar que sean
  artefactos.

#### ¿Cuánto cambian los resultados?

Esta es la pregunta que decide el tratamiento. Para cada criterio se calcula el mayor cambio relativo de las medianas
por etapa y canal si se excluyeran esas filas.

In [20]:
sensitivity = outliers.exclusion_sensitivity(tidy, {
    "Tukey entre registros": tukey_between,
    "Tukey dentro del registro": tidy["is_stat_outlier"],
    "Saturación": is_clipped,
    "Consenso multivariado": consensus,
})
display(sensitivity.round(2))

,filas excluidas,% excluido,Δ% sample_entropy,Δ% spectral_entropy,Δ% delta_rel,Δ% alpha_rel
criterio,,,,,,
Tukey entre registros,6932,15.12,6.42,3.08,3.91,16.76
Tukey dentro del registro,4658,10.16,3.53,1.95,8.75,7.10
Saturación,90,0.20,0.55,0.15,0.26,1.37
Consenso multivariado,83,0.18,0.19,0.14,0.19,0.75


Excluir por Tukey quitaría 10–15 % de los datos y movería `alpha_rel` hasta 17 %. Esto confirma que eliminarlos
**cambiaría los resultados**, sin ninguna evidencia de que esas filas estén mal. Excluir por saturación o por
consenso quita menos de 0.2 % de las filas y mueve las medianas menos de 1.4 %.

#### Resumen: cómo se trata cada caso

| Caso | Situación en estos datos | Tratamiento actual | Fundamento |
|---|---|---|---|
| Nulos en features | 0 | NaN sin imputar | Una feature no calculable no se inventa |
| Nulos en flags | Estructurales | Se conservan | Significan "sin marca" |
| Etapas `?` y `M` | 176 epochs | Conservadas como UNK/MOVE, sin imputar | Imputar sería fabricar una etiqueta |
| Ceros exactos | Solo índices válidos | Sin tratamiento | No representan faltantes |
| Amplitud baja | Concentrada en SC4221 Pz-Oz | Se conserva, documentada | Usar medidas relativas o normalizar por registro |
| Saturación | 90 filas (0.19 %) | Artefacto `clipped`: se conserva y se excluye por defecto | Criterio físico directo; no afecta a N3 |
| Outliers univariados | 10.2 % | Tukey dentro de cada noche: se marcan, nunca se eliminan | Excluirlos cambia los resultados sin justificación |
| Outliers multivariados | 85 filas de consenso | Solo diagnóstico | Sin etiqueta, no se pueden confirmar |

Las medianas por etapa, usadas en todo el análisis, son robustas: por eso el resultado casi no depende de estas filas.

## Extracción de features

Para cada epoch de 30 s (3000 muestras a 100 Hz) y cada canal:

| Feature | Definición | Unidad | Implementación |
|---|---|---|---|
| PSD | Welch: ventana Hann de 4 s (400 muestras), 50 % de solapamiento, detrend constante. Resolución 0.25 Hz | µV²/Hz | `scipy.signal.welch` |
| Potencia absoluta de banda | $P_b = \sum_{f_1 \le f < f_2} \mathrm{PSD}(f)\,\Delta f$ | µV² | Propia (suma de la PSD) |
| Bandas | delta 0.5–4, theta 4–8, alfa 8–12, sigma 12–16, beta 16–30 Hz | Hz | `config.FREQ_BANDS` |
| Potencia total | $P_{tot} = \sum_{0.5 \le f < 30} \mathrm{PSD}(f)\,\Delta f$ | µV² | Propia |
| Potencia relativa | $P_b / P_{tot}$ | — | Propia |
| Spectral Entropy | $-\sum_f p_f \log_2 p_f / \log_2 N_f$, con $p_f$ = PSD de Welch normalizada | [0, 1] | `antropy.spectral_entropy` |
| Sample Entropy | $-\ln(A/B)$; $B$ = pares de subsecuencias de largo $m=2$ con distancia de Chebyshev ≤ $r = 0.2\,\mathrm{SD}$; $A$ = ídem con $m+1$ | — | `antropy.sample_entropy` |
| RMS | $\sqrt{\overline{x^2}}$ | µV | NumPy |
| Pico a pico | $\max x - \min x$ | µV | NumPy |

Decisiones:

- **Intervalos semiabiertos y regla del rectángulo**: las bandas son disjuntas, suman exactamente la potencia total y
  la potencia total es consistente con la varianza (Parseval). Integrar con Simpson (como hace YASA) da una delta
  absoluta ~10 % menor, porque pondera menos el extremo de 0.5 Hz, donde la PSD es máxima. Por eso los valores
  **absolutos** dependen de la convención de integración y no deben compararse con trabajos que usen otra; las
  **relativas** no tienen este problema.
- **Las entropías no se implementan a mano**: se usa `antropy`, una librería mantenida y validada.
- **Spectral Entropy** se calcula sobre la PSD completa (0–50 Hz) de la señal ya filtrada: `antropy` no admite
  restringir el rango, y por encima de 30 Hz la potencia filtrada es despreciable.
- **Potencia absoluta y relativa no son intercambiables**: la absoluta depende de la amplitud global (que varía entre
  sujetos y canales); la relativa describe la *forma* del espectro.
- **Sample Entropy y Spectral Entropy miden cosas distintas**: la primera mide la predecibilidad temporal de la
  forma de onda; la segunda, cuán repartida está la potencia en frecuencia.

In [21]:
feature_cols = features.FEATURE_COLUMNS
display(tidy[feature_cols].describe().T.round(3))

,count,mean,std,min,25%,50%,75%,max
delta_abs,46212.0,142.803,257.130,0.794,22.611,52.728,143.185,5787.541
theta_abs,46212.0,15.309,18.833,0.036,5.384,9.779,17.922,467.152
alpha_abs,46212.0,8.301,11.898,0.007,2.487,4.650,9.151,191.058
sigma_abs,46212.0,4.378,8.756,0.002,1.054,2.209,4.553,152.353
beta_abs,46212.0,10.225,44.075,0.017,0.884,2.015,5.137,805.113
delta_rel,46212.0,0.700,0.196,0.047,0.597,0.744,0.852,0.995
theta_rel,46212.0,0.127,0.076,0.002,0.071,0.111,0.163,0.625
alpha_rel,46212.0,0.081,0.097,0.001,0.028,0.052,0.093,0.823
sigma_rel,46212.0,0.035,0.029,0.000,0.013,0.026,0.050,0.241
beta_rel,46212.0,0.058,0.081,0.000,0.012,0.026,0.064,0.611


## Tidy Data

La tabla procesada cumple los principios de *tidy data*:

- cada **fila** es una observación: **un epoch de 30 s de un canal EEG**;
- cada **columna** es una variable: identificadores, contexto, features y flags;
- cada **tabla** es un tipo de unidad observacional. Los epochs completos del hipnograma y los registros van en
  tablas aparte (`epochs.parquet`, `recordings.parquet`).

Se guarda en **Parquet** (`data/processed/features_tidy.parquet`): es un formato columnar, comprimido, que conserva
los tipos (categorías, booleanos, float64) y se lee mucho más rápido que un CSV.

In [22]:
display(tidy.head())
print(tidy.dtypes.to_string())
print(f"\nMemoria: {tidy.memory_usage(deep=True).sum() / 1e6:.1f} MB · "
      f"Parquet: {pipeline.TIDY_PATH.stat().st_size / 1e6:.1f} MB")

,subject_id,recording_id,night,epoch_number,start_time_sec,sleep_stage,stage_original,channel,delta_abs,theta_abs,alpha_abs,sigma_abs,beta_abs,delta_rel,theta_rel,alpha_rel,sigma_rel,beta_rel,total_power,sample_entropy,spectral_entropy,rms,peak_to_peak,is_artifact,artifact_reason,is_stat_outlier,outlier_features
0,5,SC4051,1,1229,36870.0,W,Sleep stage W,EEG Fpz-Cz,339.451321,25.669718,7.327997,1.366019,2.902387,0.901077,0.068141,0.019452,0.003626,0.007704,376.717441,0.600990,0.528237,19.577309,184.263794,False,None,False,None
1,5,SC4051,1,1229,36870.0,W,Sleep stage W,EEG Pz-Oz,7.966459,2.397041,3.808577,0.567212,1.234000,0.498736,0.150066,0.238434,0.035510,0.077254,15.973289,1.371056,0.722438,4.281704,27.830428,False,None,True,total_power;sample_entropy
2,5,SC4051,1,1230,36900.0,W,Sleep stage W,EEG Fpz-Cz,583.105163,19.929555,12.345279,1.494983,3.265508,0.940279,0.032137,0.019907,0.002411,0.005266,620.140488,0.477946,0.455049,28.128335,198.645905,False,None,False,None
3,5,SC4051,1,1230,36900.0,W,Sleep stage W,EEG Pz-Oz,11.777228,3.339115,24.293218,0.804826,1.905447,0.279612,0.079277,0.576764,0.019108,0.045239,42.119833,1.033136,0.636993,7.171573,57.623674,False,None,False,None
4,5,SC4051,1,1231,36930.0,W,Sleep stage W,EEG Fpz-Cz,392.455529,23.757771,14.714330,2.507860,5.644348,0.893814,0.054108,0.033512,0.005712,0.012855,439.079838,0.643312,0.513352,23.646409,142.768749,False,None,False,None


subject_id             int64
recording_id          object
night                  int64
epoch_number           int64
start_time_sec       float64
sleep_stage         category
stage_original        object
channel             category
delta_abs            float64
theta_abs            float64
alpha_abs            float64
sigma_abs            float64
beta_abs             float64
delta_rel            float64
theta_rel            float64
alpha_rel            float64
sigma_rel            float64
beta_rel             float64
total_power          float64
sample_entropy       float64
spectral_entropy     float64
rms                  float64
peak_to_peak         float64
is_artifact             bool
artifact_reason       object
is_stat_outlier         bool
outlier_features      object

Memoria: 15.1 MB · Parquet: 7.0 MB


## SQLite OLTP

`data/database/sleep_oltp.sqlite` se reconstruye completo en cada ejecución: es un artefacto derivado de
`data/processed/`, y reconstruirlo garantiza que no queden filas obsoletas.

```
subject 1──N recording 1──N epoch N──1 hypnogram_label N──1 sleep_stage
                              │
                              1──N eeg_feature N──1 channel
```

Restricciones definidas:

- **PK** en todas las tablas; `eeg_feature` usa la clave compuesta (epoch_id, channel_id).
- **FK** activadas con `PRAGMA foreign_keys = ON`, que en SQLite están **desactivadas por defecto**.
- **UNIQUE** en (subject_id, night), (recording_id, epoch_number), nombres de canal, códigos de etapa y archivos.
- **NOT NULL** en todo identificador y atributo obligatorio. Las features admiten NULL a propósito, porque una
  feature no calculable es un faltante legítimo.
- **CHECK** de dominios: sexo ∈ {F, M}, potencias ≥ 0, Spectral Entropy ∈ [0, 1], y coherencia entre
  `is_artifact` y `artifact_reason`.
- **Índices** sobre las FK más consultadas.

In [23]:
database_oltp.build_oltp(tidy, recordings)
display(database_oltp.table_counts().to_frame())
show_checks(validation.validate_oltp(tidy))

INFO src.database_oltp: OLTP creada: data/database/sleep_oltp.sqlite


,rows
subject,10
recording,20
channel,2
sleep_stage,7
hypnogram_label,8
epoch,23106
eeg_feature,46212


  ✔ PRAGMA foreign_keys = ON
  ✔ PRAGMA foreign_key_check sin violaciones
  ✔ recording: 20 filas
  ✔ epoch: 23106 filas = epochs de la tidy
  ✔ eeg_feature: 46212 filas = filas de la tidy
  ✔ Potencia relativa recalculada en la vista coincide (max |dif| = 1.8e-15)


In [24]:
# Demostración: las restricciones se aplican (cada intento se revierte).
import sqlite3
with database_oltp.connect() as conn:
    for description, sql in [
        ("epoch de un registro inexistente", "INSERT INTO epoch (recording_id, epoch_number, label_id) VALUES ('SC9999', 0, 1)"),
        ("(epoch, canal) duplicado", "INSERT INTO eeg_feature (epoch_id, channel_id, is_artifact) VALUES (1, 1, 0)"),
        ("artefacto sin motivo", "INSERT INTO eeg_feature (epoch_id, channel_id, is_artifact) VALUES (1, 2, 1)"),
    ]:
        try:
            conn.execute(sql)
            print(f"  ✘ {description}: NO fue rechazado")
        except sqlite3.IntegrityError as error:
            print(f"  ✔ {description}: rechazado ({error})")
    conn.rollback()
conn.close()

  ✔ epoch de un registro inexistente: rechazado (FOREIGN KEY constraint failed)
  ✔ (epoch, canal) duplicado: rechazado (UNIQUE constraint failed: eeg_feature.epoch_id, eeg_feature.channel_id)
  ✔ artefacto sin motivo: rechazado (CHECK constraint failed: (is_artifact = 0 AND artifact_reason IS NULL) OR (is_artifact = 1 AND artifact_reason IS NOT NULL))


## Normalización

**1NF.** Todos los atributos son atómicos (un valor por celda; no hay listas ni grupos repetidos) y cada tabla tiene
clave primaria. Las bandas de frecuencia se modelaron como columnas fijas (`delta_abs`, …) y no como un grupo
repetido: son un conjunto cerrado y conocido de variables de la misma observación.

**2NF.** No hay dependencias parciales de una clave compuesta:

- en `eeg_feature` (PK = epoch_id + channel_id), cada feature depende del par completo: la potencia delta del epoch
  100 en Fpz-Cz no está determinada solo por el epoch ni solo por el canal;
- `start_sec = epoch_number × 30` dependería solo de `epoch_number`, que es parte de la clave candidata
  (recording_id, epoch_number). Por eso **no se almacena** y se calcula en la vista `v_epoch`;
- el flag de artefacto depende del canal (un parpadeo afecta a Fpz-Cz y no a Pz-Oz), por eso vive en `eeg_feature`
  y no en `epoch`.

**3NF.** No hay dependencias transitivas entre atributos no clave:

- la etiqueta original del hipnograma determina la etapa ("Sleep stage 4" → N3). Guardar ambas en `epoch` crearía
  `epoch → etiqueta → etapa`; por eso existe `hypnogram_label`, y `epoch` solo referencia la etiqueta;
- edad y sexo dependen del sujeto, no del registro: están en `subject`;
- la potencia total y las relativas se derivan de las absolutas (total = Σ bandas; relativa = banda / total).
  Almacenarlas sería redundante; las calcula la vista `v_eeg_feature`. La validación anterior confirma que la vista
  reproduce los valores de la tabla tidy (diferencia ~1e-15);
- los outliers estadísticos dependen de la distribución de cada noche, no de la fila: no son atributos de la
  observación y se calculan en el ETL hacia el Data Warehouse.

## Data Warehouse

```
SQLite OLTP (3NF)
    ↓  Extract: una consulta SQL con JOIN sobre las vistas del OLTP
    ↓  Transform: dimensiones con surrogate keys, outliers estadísticos, tiempo desde el inicio del sueño
    ↓  Load: dimensiones primero, luego hechos (orden requerido por las FK)
SQLite Data Warehouse (estrella)
```

El ETL lee **del OLTP**, no de los Parquet: el OLTP es la fuente de verdad transaccional y el DW, su modelo analítico
derivado.

In [25]:
dw_counts = database_dw.build_dw()
display(pd.Series(dw_counts, name="filas").to_frame())
show_checks(validation.validate_dw())

INFO src.database_dw: DW creado: data/database/sleep_dw.sqlite


,filas
dim_subject,10
dim_recording,20
dim_channel,2
dim_sleep_stage,8
fact_eeg_epoch,46212


  ✔ DW: PRAGMA foreign_key_check sin violaciones
  ✔ fact_eeg_epoch: 46212 filas = eeg_feature del OLTP
  ✔ Grano respetado: (registro, epoch, canal) único en la tabla de hechos
  ✔ Cada registro pertenece a un único sujeto
  ✔ 8 consultas analíticas reproducibles


## Esquema estrella

**Grano de `fact_eeg_epoch`:** *una observación de features EEG correspondiente a un epoch de 30 segundos, un canal,
un sujeto y un registro.*

```
                    dim_subject
                         │
   dim_recording ── fact_eeg_epoch ── dim_channel
                         │
                  dim_sleep_stage
```

| Tabla | Contenido |
|---|---|
| `fact_eeg_epoch` | Claves de las 4 dimensiones, `epoch_number` (dimensión degenerada), `start_sec`, `hours_since_sleep_onset`, las 15 métricas y los flags de calidad |
| `dim_subject` | Edad, sexo, grupo etario |
| `dim_recording` | Noche, inicio, apagado de luces, duración total y analizada |
| `dim_channel` | Nombre, nombre corto, ubicación |
| `dim_sleep_stage` | Grano = etiqueta original del hipnograma, con su código AASM (S3 y S4 → N3): la jerarquía queda desnormalizada en la dimensión |

Decisiones:

- **Surrogate keys** enteras y asignadas en un orden determinístico, para que el DW sea idéntico en cada ejecución.
- **Las dimensiones no se referencian entre sí** (eso sería un copo de nieve): la tabla de hechos tiene las 4 claves.
- **Redundancia deliberada**: a diferencia del OLTP, se materializan las potencias total y relativas, el tiempo desde
  el inicio del sueño y los flags de outlier. Así las consultas analíticas necesitan un solo JOIN por dimensión.
- `hours_since_sleep_onset` permite comparar noches que empiezan a horas distintas.

In [26]:
with database_dw.connect() as conn:
    for table in ["dim_subject", "dim_recording", "dim_channel", "dim_sleep_stage"]:
        display(Markdown(f"**{table}**"))
        display(pd.read_sql(f"SELECT * FROM {table}", conn))
conn.close()

**dim_subject**

,subject_key,subject_id,age,sex,age_group
0,1,5,28,F,<40
1,2,14,27,M,<40
2,3,20,51,F,40-59
3,4,22,56,F,40-59
4,5,33,60,M,60-79
5,6,38,51,M,40-59
6,7,40,67,F,60-79
7,8,50,71,M,60-79
8,9,65,88,F,80+
9,10,76,90,M,80+


**dim_recording**

,recording_key,recording_id,night,start_datetime,lights_off,sampling_frequency_hz,psg_duration_h,analyzed_duration_h
0,1,SC4051,1,1989-04-26T16:25:00,01:22:00,100.0,22.683333,5.600000
1,2,SC4052,2,1989-04-27T15:42:00,00:35:00,100.0,23.383333,10.400000
2,3,SC4141,1,1989-06-19T16:05:00,22:55:00,100.0,22.966667,8.366667
3,4,SC4142,2,1989-06-20T15:33:00,23:22:00,100.0,23.116667,7.933333
4,5,SC4201,1,1989-05-08T16:18:00,23:10:00,100.0,23.366667,8.525000
5,6,SC4202,2,1989-05-09T16:09:00,23:15:00,100.0,22.250000,8.508333
6,7,SC4221,1,1989-05-24T16:55:00,23:47:00,100.0,22.500000,9.158333
7,8,SC4222,2,1989-05-25T16:30:00,23:14:00,100.0,23.000000,9.233333
8,9,SC4331,1,1990-03-19T15:15:00,22:58:00,100.0,23.416667,8.233333
9,10,SC4332,2,1990-03-20T15:20:00,22:55:00,100.0,23.000000,8.825000


**dim_channel**

,channel_key,channel_name,short_name,location
0,1,EEG Fpz-Cz,Fpz-Cz,Bipolar frontal-central (Fpz referenced to Cz)
1,2,EEG Pz-Oz,Pz-Oz,Bipolar parietal-occipital (Pz referenced to Oz)


**dim_sleep_stage**

,stage_key,original_description,stage_code,stage_name,is_scored,stage_order
0,1,Sleep stage W,W,Wake,1,1
1,2,Sleep stage 1,N1,NREM 1,1,2
2,3,Sleep stage 2,N2,NREM 2,1,3
3,4,Sleep stage 3,N3,NREM 3 (slow-wave sleep),1,4
4,5,Sleep stage 4,N3,NREM 3 (slow-wave sleep),1,4
5,6,Sleep stage R,REM,Rapid eye movement,1,5
6,7,Movement time,MOVE,Movement time (unscored),0,6
7,8,Sleep stage ?,UNK,Unknown (unscored),0,7


## Consultas SQL

Las consultas están en `src/queries.py`. Comparten dos convenciones:

- `:include_artifacts` (0/1) controla si se incluyen las observaciones con artefacto;
- las comparaciones entre etapas usan solo etapas puntuadas (`is_scored = 1`). Las consultas de calidad (conteos y
  porcentaje de artefactos) describen todo el conjunto.

Como ejemplo, la consulta 1 completa:

In [27]:
print(queries.QUERIES["sample_entropy_by_stage"])


        SELECT st.stage_code, c.short_name AS channel,
               COUNT(f.sample_entropy) AS n, AVG(f.sample_entropy) AS mean_sample_entropy,
               MIN(f.sample_entropy) AS min_value, MAX(f.sample_entropy) AS max_value
        
FROM fact_eeg_epoch f
JOIN dim_subject     su ON su.subject_key   = f.subject_key
JOIN dim_recording   r  ON r.recording_key  = f.recording_key
JOIN dim_channel     c  ON c.channel_key    = f.channel_key
JOIN dim_sleep_stage st ON st.stage_key     = f.stage_key
 WHERE st.is_scored = 1 AND (:include_artifacts = 1 OR f.is_artifact = 0)
        GROUP BY st.stage_code, st.stage_order, c.short_name
        ORDER BY st.stage_order, channel


In [28]:
# 1. Sample Entropy promedio por etapa
q1 = queries.run_query("sample_entropy_by_stage")
display(q1.round(3))

,stage_code,channel,n,mean_sample_entropy,min_value,max_value
0,W,Fpz-Cz,6479,1.021,0.030,2.069
1,W,Pz-Oz,6527,1.317,0.041,2.038
2,N1,Fpz-Cz,2160,1.031,0.067,1.771
3,N1,Pz-Oz,2161,0.985,0.180,1.879
4,N2,Fpz-Cz,9249,0.792,0.062,1.667
5,N2,Pz-Oz,9244,0.826,0.199,1.856
6,N3,Fpz-Cz,1317,0.560,0.274,1.175
7,N3,Pz-Oz,1317,0.618,0.361,1.554
8,REM,Fpz-Cz,3656,1.044,0.083,1.802
9,REM,Pz-Oz,3660,1.026,0.106,1.597


In [29]:
# 2. Spectral Entropy promedio por etapa
q2 = queries.run_query("spectral_entropy_by_stage")
display(q2.round(3))

,stage_code,channel,n,mean_spectral_entropy,min_value,max_value
0,W,Fpz-Cz,6479,0.615,0.262,0.918
1,W,Pz-Oz,6527,0.700,0.234,0.918
2,N1,Fpz-Cz,2160,0.635,0.271,0.861
3,N1,Pz-Oz,2161,0.648,0.281,0.887
4,N2,Fpz-Cz,9249,0.583,0.263,0.836
5,N2,Pz-Oz,9244,0.620,0.277,0.858
6,N3,Fpz-Cz,1317,0.490,0.338,0.671
7,N3,Pz-Oz,1317,0.537,0.410,0.719
8,REM,Fpz-Cz,3656,0.627,0.245,0.863
9,REM,Pz-Oz,3660,0.667,0.281,0.816


In [30]:
# 3. Potencia relativa de cada banda por etapa
q3 = queries.run_query("relative_band_power_by_stage")
display(q3.round(3))

,stage_code,channel,n,delta,theta,alpha,sigma,beta
0,W,Fpz-Cz,6479,0.706,0.088,0.081,0.031,0.093
1,W,Pz-Oz,6527,0.465,0.132,0.186,0.060,0.158
2,N1,Fpz-Cz,2160,0.698,0.119,0.083,0.038,0.061
3,N1,Pz-Oz,2161,0.624,0.186,0.106,0.034,0.050
4,N2,Fpz-Cz,9249,0.808,0.098,0.042,0.031,0.020
5,N2,Pz-Oz,9244,0.732,0.158,0.056,0.033,0.022
6,N3,Fpz-Cz,1317,0.916,0.053,0.018,0.009,0.005
7,N3,Pz-Oz,1317,0.854,0.102,0.025,0.011,0.008
8,REM,Fpz-Cz,3656,0.716,0.133,0.065,0.030,0.056
9,REM,Pz-Oz,3660,0.632,0.186,0.104,0.034,0.044


In [31]:
# 4. Comparación entre canales
q4 = queries.run_query("channel_comparison")
display(q4.round(3))

,channel,stage_code,n,sample_entropy,spectral_entropy,delta_rel,alpha_rel,total_power,rms
0,Fpz-Cz,W,6479,1.021,0.615,0.706,0.081,488.921,20.948
1,Pz-Oz,W,6527,1.317,0.700,0.465,0.186,99.933,9.477
2,Fpz-Cz,N1,2160,1.031,0.635,0.698,0.083,121.109,10.546
3,Pz-Oz,N1,2161,0.985,0.648,0.624,0.106,43.736,6.267
4,Fpz-Cz,N2,9249,0.792,0.583,0.808,0.042,187.101,13.472
5,Pz-Oz,N2,9244,0.826,0.620,0.732,0.056,67.983,8.082
6,Fpz-Cz,N3,1317,0.560,0.490,0.916,0.018,624.560,25.440
7,Pz-Oz,N3,1317,0.618,0.537,0.854,0.025,191.684,13.691
8,Fpz-Cz,REM,3656,1.044,0.627,0.716,0.065,85.540,8.934
9,Pz-Oz,REM,3660,1.026,0.667,0.632,0.104,32.347,5.600


In [32]:
# 5. Comparación entre sujetos
q5 = queries.run_query("subject_comparison")
display(q5.round(3).head(20))

,subject_id,age,sex,stage_code,channel,n,sample_entropy,spectral_entropy,delta_rel
0,5,28,F,W,Fpz-Cz,413,0.814,0.560,0.707
1,5,28,F,W,Pz-Oz,430,0.908,0.593,0.501
2,5,28,F,N1,Fpz-Cz,158,1.001,0.666,0.601
3,5,28,F,N1,Pz-Oz,158,0.846,0.652,0.528
4,5,28,F,N2,Fpz-Cz,833,0.815,0.629,0.735
5,5,28,F,N2,Pz-Oz,833,0.773,0.624,0.691
6,5,28,F,N3,Fpz-Cz,249,0.567,0.519,0.903
7,5,28,F,N3,Pz-Oz,249,0.567,0.518,0.882
8,5,28,F,REM,Fpz-Cz,248,0.929,0.629,0.687
9,5,28,F,REM,Pz-Oz,248,0.932,0.673,0.574


In [33]:
# 6. Cantidad de epochs por etapa
q6 = queries.run_query("epochs_by_stage")
display(q6.round(3))

,stage_code,n_epochs,percent,hours
0,W,6536,28.29,54.47
1,N1,2161,9.35,18.01
2,N2,9254,40.05,77.12
3,N3,1317,5.70,10.97
4,REM,3662,15.85,30.52
5,MOVE,22,0.10,0.18
6,UNK,154,0.67,1.28


In [34]:
# 7. Porcentaje de epochs marcados como artefactos
q7 = queries.run_query("artifact_percentage")
display(q7.round(3))

,recording_id,channel,n_observations,n_artifacts,artifact_percent,n_stat_outliers,stat_outlier_percent
0,SC4051,Fpz-Cz,672,2,0.30,57,8.48
1,SC4051,Pz-Oz,672,0,0.00,96,14.29
2,SC4052,Fpz-Cz,1248,15,1.20,104,8.33
3,SC4052,Pz-Oz,1248,0,0.00,135,10.82
4,SC4141,Fpz-Cz,1004,0,0.00,99,9.86
5,SC4141,Pz-Oz,1004,0,0.00,100,9.96
6,SC4142,Fpz-Cz,952,0,0.00,98,10.29
7,SC4142,Pz-Oz,952,0,0.00,110,11.55
8,SC4201,Fpz-Cz,1023,34,3.32,83,8.11
9,SC4201,Pz-Oz,1023,2,0.20,87,8.50


In [35]:
# 8. Evolución temporal durante la noche
q8 = queries.run_query("temporal_evolution")
display(q8.round(3).head(20))

,recording_id,channel,hour_bin,n,sample_entropy,spectral_entropy,delta_rel,pct_n3,pct_rem
0,SC4051,Fpz-Cz,-1,60,0.903,0.575,0.644,0.0,0.0
1,SC4051,Fpz-Cz,0,120,0.653,0.570,0.834,58.3,0.0
2,SC4051,Fpz-Cz,1,120,0.784,0.624,0.712,10.8,36.7
3,SC4051,Fpz-Cz,2,120,0.929,0.638,0.590,0.8,0.0
4,SC4051,Fpz-Cz,3,120,0.659,0.562,0.825,42.5,20.0
5,SC4051,Fpz-Cz,4,119,0.720,0.555,0.777,0.0,0.0
6,SC4051,Fpz-Cz,5,11,0.384,0.520,0.926,0.0,0.0
7,SC4051,Pz-Oz,-1,60,0.902,0.572,0.306,0.0,0.0
8,SC4051,Pz-Oz,0,120,0.652,0.574,0.798,58.3,0.0
9,SC4051,Pz-Oz,1,120,0.823,0.647,0.634,10.8,36.7


In [36]:
# Reproducibilidad: todas las consultas devuelven exactamente lo mismo al repetirse.
for name in queries.QUERIES:
    pd.testing.assert_frame_equal(queries.run_query(name), queries.run_query(name))
print(f"✔ {len(queries.QUERIES)} consultas reproducibles")

✔ 8 consultas reproducibles


## EDA

Todas las figuras se construyen con datos del **Data Warehouse** (vía `src/queries.py`), no desde los EDF. Cada una
responde una pregunta concreta.

**Paleta**: W naranja; N1→N3 una rampa azul de claro a oscuro (profundidad creciente del sueño); REM aqua. Se validó
para visión normal y daltonismo (protanopía/deuteranopía) en todos los pares de colores.

In [37]:
obs = queries.load_observations()
print(f"Observaciones cargadas desde el DW: {len(obs):,}")

Observaciones cargadas desde el DW: 46,212


**¿Cuánto tiempo pasa en cada etapa dentro de la ventana de análisis?**

In [38]:
plots.stage_distribution(queries.run_query("epochs_by_stage")).show()

**¿Cómo cambia la Sample Entropy entre etapas, y es igual en ambos canales?**

In [39]:
plots.metric_by_stage(obs[~obs["is_artifact"]], "sample_entropy").show()

**¿Cómo cambia la Spectral Entropy entre etapas?**

In [40]:
plots.metric_by_stage(obs[~obs["is_artifact"]], "spectral_entropy").show()

**¿Qué bandas dominan cada etapa? (canal posterior, donde el alfa de vigilia es más visible)**

In [41]:
rel = queries.run_query("relative_band_power_by_stage")
plots.band_power_by_stage(rel, "Pz-Oz").show()

**¿En qué etapa es cada banda relativamente más fuerte? El color es el z-score dentro de cada banda; el texto, la potencia relativa media.**

In [42]:
for ch in ["Fpz-Cz", "Pz-Oz"]:
    plots.stage_band_heatmap(rel, ch).show()

**¿Se asocian la actividad delta y la complejidad? ¿La relación depende de la etapa?**

In [43]:
clean = obs[~obs["is_artifact"]]
for ch in ["Fpz-Cz", "Pz-Oz"]:
    plots.delta_vs_entropy(clean, ch, max_points_per_stage=1500).show()

**¿Difieren el canal frontal y el posterior?**

In [44]:
plots.channel_comparison(queries.run_query("channel_comparison"), "sample_entropy").show()
plots.channel_comparison(queries.run_query("channel_comparison"), "alpha_rel").show()

**¿El patrón entre etapas se repite en cada sujeto, o lo produce un sujeto en particular?**

In [45]:
plots.subject_comparison(queries.run_query("subject_comparison"), "sample_entropy", "Pz-Oz").show()

**¿Cómo evolucionan la delta relativa y la complejidad a lo largo de la noche?**

In [46]:
evo = queries.run_query("temporal_evolution")
plots.temporal_evolution(evo, "delta_rel", "Fpz-Cz").show()
plots.temporal_evolution(evo, "sample_entropy", "Fpz-Cz").show()

**¿Cuán fuerte es la relación lineal entre las métricas, y cuánto de ella se explica por la etapa?** Se compara la
correlación de Spearman global con la mediana de las correlaciones dentro de cada etapa: si la correlación global es
mucho mayor, gran parte de la asociación se debe a que ambas variables cambian con la etapa (confusión), no a una
relación dentro de la etapa.

In [47]:
pairs = [("delta_rel", "sample_entropy"), ("delta_rel", "spectral_entropy"), ("sample_entropy", "spectral_entropy")]
scored = obs[obs["is_scored"] & ~obs["is_artifact"]]
rows = []
for ch, g in scored.groupby("channel"):
    for a, b in pairs:
        within = g.groupby("stage_code").apply(lambda s: s[a].corr(s[b], method="spearman"), include_groups=False)
        rows.append({"canal": ch, "par": f"{a} ~ {b}", "ρ global": g[a].corr(g[b], method="spearman"),
                     "ρ dentro de etapa (mediana)": within.median()})
display(pd.DataFrame(rows).round(2))

,canal,par,ρ global,ρ dentro de etapa (mediana)
0,Fpz-Cz,delta_rel ~ sample_entropy,-0.87,-0.85
1,Fpz-Cz,delta_rel ~ spectral_entropy,-0.93,-0.91
2,Fpz-Cz,sample_entropy ~ spectral_entropy,0.84,0.83
3,Pz-Oz,delta_rel ~ sample_entropy,-0.84,-0.76
4,Pz-Oz,delta_rel ~ spectral_entropy,-0.89,-0.89
5,Pz-Oz,sample_entropy ~ spectral_entropy,0.86,0.85


## Dashboard

`src/app.py` es un dashboard Streamlit que lee **solo** `data/database/sleep_dw.sqlite`.

- **Filtros**: sujeto, noche/registro, canal, etapas de sueño, incluir/excluir artefactos.
- **Contenido**: hipnograma; Sample Entropy y Spectral Entropy por etapa; potencia relativa por banda; heatmap
  etapa × banda; scatter delta vs Sample Entropy coloreado por etapa.

Ejecución local, desde la raíz del repositorio:

```bash
streamlit run src/app.py
```

En Colab, Streamlit necesita un túnel para exponer el puerto; las mismas figuras ya se muestran en la sección EDA de
este notebook. Ver el README para la opción con túnel.

## Resultados principales

**Muestra:**
- 10 sujetos (5 mujeres y 5 hombres, de 27 a 90 años) × 2 noches = 20 registros;
- 23 106 epochs analizados (≈193 h) y 46 212 observaciones epoch × canal.

Los valores citados corresponden a esta ejecución y se recalculan en las celdas anteriores. **Todos los resultados
son descriptivos.**

**1. Distribución de etapas.** Dentro de la ventana nocturna, N2 es la etapa más frecuente (≈40 % de los epochs),
seguida de W (≈28 %, que incluye los despertares nocturnos), REM (≈16 %), N1 (≈9 %) y N3 (≈6 %).

- El N3 varía mucho entre sujetos: ≈13–15 % en los sujetos de 27–28 años, y casi ausente (≤2 %) en varios mayores
  de 50. Un sujeto de 60 años no tiene ningún epoch N3. Es la disminución del sueño de ondas lentas con la edad,
  bien descrita en la literatura.
- Hubo 154 epochs `?` y 22 de *Movement time* dentro de la ventana. Se conservaron sin imputar.

**2. Complejidad temporal (Sample Entropy).**

- Baja con la profundidad del sueño NREM, N1 > N2 > N3 (en Pz-Oz ≈0.98 → 0.83 → 0.62). REM queda en niveles
  similares a N1 (≈1.03).
- El patrón es robusto entre sujetos: con promedios, **N1 > N2 > N3 se cumple en las 18 combinaciones sujeto × canal
  con N3** (las otras dos casi no tienen N3), y REM > N2 en las 20. Con medianas hay una excepción en cada caso, ambas
  en Pz-Oz: el sujeto 22 (N1 < N2) y el sujeto 76 (REM < N2). Excluir o no los artefactos no cambia estos conteos.
- La vigilia muestra la mayor complejidad en Pz-Oz (≈1.32). En Fpz-Cz es menor y más variable, compatible con la
  contaminación por movimientos oculares, que agrega actividad lenta y regular en el canal frontal.

**3. Dispersión espectral (Spectral Entropy).** N3 tiene la menor Spectral Entropy (≈0.49–0.54) en **las 18
combinaciones sujeto × canal con N3** usando promedios (17 de 18 con medianas; la excepción es de nuevo el sujeto 22 en
Pz-Oz): la potencia se concentra en pocas frecuencias. Las dos entropías cuentan
historias parecidas pero no idénticas, porque miden propiedades distintas de la señal.

**4. Actividad espectral.**

- La **delta relativa** crece con la profundidad del sueño: ≈0.85–0.92 en N3.
- El **alfa relativo** es máximo en vigilia en el canal posterior (Pz-Oz ≈0.19, contra ≈0.08 en Fpz-Cz), coherente
  con el ritmo alfa occipital de la vigilia relajada.
- **Theta** es relativamente más alta en N1 y REM.
- **Beta** es máxima en vigilia.

**5. Delta y complejidad.** La delta relativa y la Sample Entropy tienen una fuerte asociación negativa (Spearman
ρ ≈ −0.85). La correlación se mantiene **dentro de cada etapa** (ρ ≈ −0.63 a −0.88, salvo W en Pz-Oz con ≈ −0.42),
así que no se explica solo por el cambio de etapa: dentro de un mismo estado, los epochs con más potencia lenta son
más predecibles. Es una asociación entre dos descriptores de la misma señal, **no una relación causal**.

**6. Evolución nocturna.** El N3 se concentra en las primeras horas tras el inicio del sueño (≈14–17 % en las horas
0–2) y casi desaparece después de la hora 5. El REM, en cambio, aumenta hacia la segunda mitad de la noche (≈30 % en
las horas 5–6). Es el patrón clásico de la arquitectura del sueño. Al final de la noche quedan pocos registros (las
noches tienen duraciones distintas), por eso el gráfico solo muestra las horas que cubren al menos la mitad de ellos.

**7. Calidad de datos.**

- **Artefactos físicos**: 90 observaciones (0.19 %) con saturación del conversor, 66 de ellas en W y ninguna en N3. Los
  artefactos sutiles (oculares, musculares) no se detectan con estos criterios: es una limitación, no una prueba de
  ausencia de artefactos.
- **Outliers estadísticos**: 10.2 % de las observaciones según la regla de Tukey dentro de cada noche, entre 3.8 % y
  19.9 % por registro y canal. Son informativos y no se eliminan.
- **Comparar registros también es útil.** Con Tukey entre registros, SC4221 tiene 52 % de filas marcadas: esa regla no
  sirve para encontrar epochs raros, pero sí para detectar un registro distinto.
- **El caso SC4221 / Pz-Oz** muestra para qué sirve esa comparación. El canal posterior de esa noche tiene una amplitud
  ~4 veces menor que la otra noche del mismo sujeto (RMS mediana ≈1.9 µV contra ≈6.4 µV), mientras que Fpz-Cz es
  normal. Esto sugiere un problema técnico de registro (contacto del electrodo o ganancia) más que fisiología.
  - No llega a ser una señal plana, por lo que no se marca como artefacto.
  - Sus métricas **relativas** e invariantes a la escala (potencia relativa, Sample Entropy con r = 0.2·SD,
    Spectral Entropy) siguen siendo interpretables.
  - Sus métricas **absolutas** no lo son.
  - El diagnóstico no eliminó nada, pero permitió detectar el caso y documentarlo: un outlier estadístico es una
    señal para revisar, no un criterio de exclusión.

**En síntesis:** al profundizarse el sueño NREM, el EEG se vuelve más lento (más delta), más regular (menor Sample
Entropy) y con la potencia más concentrada (menor Spectral Entropy). REM y N1 se parecen a la vigilia en complejidad,
pero con menos alfa posterior. El patrón se repite en sujetos de 27 a 90 años, aunque la cantidad de sueño profundo
cambia mucho con la edad. Estas observaciones coinciden con lo conocido sobre la fisiología del sueño, lo que funciona
también como validación del pipeline.

## Limitaciones

- **Muestra pequeña**: 10 sujetos sanos de un único estudio, estratificados por sexo y edad (27–90 años). Con un
  sujeto por estrato no se pueden separar los efectos de edad y sexo de la variabilidad individual. Los resultados no
  se generalizan a patologías.
- **Edad como fuente de variabilidad**: el sueño de ondas lentas disminuye con la edad, así que los promedios por
  etapa mezclan sujetos con arquitecturas de sueño muy distintas. La comparación entre sujetos lo hace visible.
- **Sin inferencia estadística**: los epochs de una misma noche no son independientes (autocorrelación temporal y
  anidamiento sujeto/noche). Un análisis inferencial necesitaría modelos mixtos con sujeto como efecto aleatorio.
- **Puntuación R&K**: los hipnogramas siguen Rechtschaffen & Kales (1968); unir S3+S4 en N3 aproxima AASM, pero
  los criterios no son idénticos.
- **Dos canales bipolares**: Fpz-Cz y Pz-Oz no permiten análisis topográfico. Fpz-Cz es sensible a la actividad
  ocular, que cae en la banda delta, sobre todo en vigilia.
- **Calidad por canal y registro**: no se evalúa automáticamente si un canal entero tiene una amplitud anómala. El
  caso SC4221 / Pz-Oz (amplitud ~4 veces menor que en la otra noche) se detectó al comparar las tasas de
  outliers entre registros y se documentó, pero sus métricas absolutas no son comparables con las del resto.
- **Detección de artefactos conservadora**: los criterios físicos (saturación, señal plana, no finitos) no detectan
  artefactos sutiles, como movimientos oculares o EMG de baja amplitud. No se aplicaron métodos avanzados (ICA,
  regresión de EOG) para evitar sobreprocesar la señal.
- **Convenciones de las métricas**: la potencia absoluta depende del método de integración, y la Sample Entropy
  depende de m, r y la longitud del epoch. Las comparaciones son válidas dentro de este pipeline, no necesariamente
  con valores publicados que usen otros parámetros.
- **Recorte de vigilia**: conservar ±30 min es una convención. Otra ventana cambiaría la proporción de W y sus
  estadísticos.

## Reproducibilidad

| Aspecto | Cómo se garantiza |
|---|---|
| Datos | Descarga programática desde la fuente pública; verificación SHA256 contra los hashes oficiales; manifest |
| Rutas | `pathlib`, relativas a la raíz del repositorio; sin rutas absolutas ni archivos privados |
| Parámetros | Centralizados en `src/config.py`; su hash se guarda con cada resultado intermedio |
| Caché | Por registro, invalidado automáticamente si cambian parámetros o versión del código |
| Aleatoriedad | La única aleatoriedad (submuestreo para el scatter) usa `RANDOM_SEED` |
| Bases | Se reconstruyen desde cero en cada ejecución; surrogate keys determinísticas |
| Consultas | Verificadas como reproducibles (mismo resultado en ejecuciones repetidas) |
| Entorno | Versiones registradas en `data/processed/environment.json` |

In [48]:
env = json.loads(pipeline.ENVIRONMENT_PATH.read_text())
display(pd.Series(env["library_versions"], name="versión").to_frame())
print("Hash de parámetros del Transform:", env["params_hash"])

,versión
python,3.13.11
numpy,2.2.6
pandas,2.2.3
scipy,1.15.3
mne,1.9.0
mne-bids,0.20.0
antropy,0.1.9
pyarrow,25.0.1
plotly,7.1.0
streamlit,1.65.0


Hash de parámetros del Transform: a9537c030c33


## Conclusiones

Se construyó un pipeline de Ingeniería de Datos reproducible de punta a punta:

- **Extract** desde una fuente pública, con integridad verificada.
- **Organización BIDS** que separa los datos crudos de su representación estandarizada.
- **Transform** documentado: epochs, recorte, filtro y features con librerías científicas mantenidas.
- **Control de calidad** explícito, que distingue datos faltantes, outliers estadísticos y artefactos de señal sin
  eliminar ni imputar información.
- **Modelado relacional 3NF**, con restricciones verificadas.
- **Data Warehouse en estrella**, alimentado por un ETL desde el OLTP.
- **Consultas SQL reproducibles**, visualizaciones y un dashboard que leen únicamente del DW.

Sobre la pregunta del caso de estudio, los datos muestran de forma consistente entre sujetos y canales que, a medida
que el sueño NREM se profundiza, la potencia se concentra en delta y la complejidad (en sus dos acepciones) disminuye.
REM y N1 se parecen más a la vigilia en complejidad, pero con menos alfa posterior que la vigilia relajada. Estas
observaciones son descriptivas y coinciden con lo esperado por la fisiología del sueño, lo que funciona también como
validación del pipeline.

### Trabajo futuro: almacenamiento de EEG continuo

Este trabajo almacena **features por epoch**: un volumen pequeño que SQLite maneja sin dificultad. Si el objetivo
fuera almacenar **señal EEG continua**, por ejemplo con adquisición en tiempo real o *wearables* de sueño que
transmiten muestras de forma permanente, convendría una base de series temporales como **TimescaleDB** (extensión de
PostgreSQL), que ofrece:

- particionado automático por tiempo (*hypertables*);
- compresión columnar;
- agregados continuos (por ejemplo, potencia por banda cada 30 s calculada en la base);
- políticas de retención.

No es necesaria para este TP, cuyo foco es el modelado OLTP/DW de métricas derivadas.